# Stage 02a — Preprocessing (review-batch validation & cleanup)

Runs **between** the human review export (`reviewed_patents_<batch>.xlsx`,
produced by the HTML wizard off `01a_wizard_feed`) and `02b_postprocessing`.

Reads the **corrected working copy** in `03c_CORRECTED_wizard_exports/`
(falling back to the frozen export, with a printed notice, for a batch that has
no copy yet). The frozen human exports in `03_HUMAN_wizard_exports/` are never
written to.

Pipeline: **load** (resolving blank `Image_Path`s in-memory via
`scripts/resolve_image_paths.py` logic — no file on disk is touched) →
**validate & clean** (ghost-block cleanup, `topType` TP→TR codebook rename,
Rule A Combined Thrust, Rule C duplicate-chain `UAVSimilar` propagation,
Rule D duplicate inheritance, Rule A1 fixed-arch `acState`→`HoverCruise`,
Rule E `acState` Ground→Other, multi-arch name suffix, then a completeness
report) → **Section 5 review queue** for the decisions that need a figure,
with Section 5c writing the confirmed ones back to `03c` →
**export** approved-only images as a timestamped
`Review_postprocess_<batch>_<timestamp>.xlsx` into
`02a_CLEANED_label_tables/` for `02b_postprocessing`.

The legacy migrations (`empKin`→`empTilts`, Draft→`acSty`, `boomXFormat`,
`fusKin` "Variable") live in **`02a_legacy.ipynb`**, which handles Batches 01
and 05 by round-tripping them through the wizard itself.

Format contract (differs from `excel_schema.py`'s source format!):
7 columns, `Value`s are `"ID — Label"` composites, M3 kinematics are
card-prefixed (`wing1_propKin`, ...), edge tags live in `META/t1EdgeTags`.


## Section 1 — Imports & Config

In [1]:
import sys
from pathlib import Path
from datetime import datetime

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import networkx as nx          # Rule C — duplicate-chain connected components
import openpyxl
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
import ipywidgets as widgets
from IPython.display import display, clear_output

from src.config_loader import load_config
import src.processor as proc   # Section 5 — parse_arch_id() for image lookup

cfg = load_config()

sheet_name = "Batch_01"   # <- the batch to preprocess

# ── Run-mode knobs (leave both at their defaults for a normal labelling run) ─
# HEADLESS: True → every interactive widget step is skipped/auto-applied so the
#   whole notebook can execute top-to-bottom unattended ("Run All" / nbconvert
#   smoke test). For real labelling leave it False.
# SMOKE_N_PATENTS: >0 → only the first N patents are loaded (fast pipeline
#   check). 0 = full batch.
import os
HEADLESS = os.environ.get("NB02A_HEADLESS", "0") == "1"
SMOKE_N_PATENTS = int(os.environ.get("NB02A_SMOKE_N", "0"))

print(f"batch: {sheet_name}   |   mode: {'HEADLESS (no widgets)' if HEADLESS else 'interactive'}"
      + (f"   |   SMOKE subset: first {SMOKE_N_PATENTS} patents" if SMOKE_N_PATENTS else ""))


batch: Batch_01   |   mode: interactive


## Section 2 — "Rearranging the excell" layout logic (reusable)

Ports `src/Rearranging th eexcell.py`'s three visual behaviors into one
function we call from Section 6:

1. **Cell merging** — for every column, consecutive rows sharing the same
   value are merged into one cell (vertical-center, wrap-text), exactly as
   the original script's "merge consecutive rows with identical values" step.
2. **Column widths** — the original hardcoded a `{'A': 18, 'B': 10, ...}`
   letter->width dict tied to one fixed 7-column layout (`Patent_ID, Section,
   Sub_Dimension, Field, Value, Source, Image_Path`). Our schema carries more
   columns (`Definition`, `Options`, `Confidence`, `Needs_Review`,
   `pre_process_flags`) and the column order can shift, so a hardcoded letter
   map would silently mis-size or ignore columns. Instead we **auto-fit**:
   each column's width is `max(min_width, min(longest_value + 2, max_width))`,
   computed from that column's actual content.
3. **Text wrap / alignment** — final pass sets `wrap_text=True` on every
   data cell, `vertical='center'` for merged cells and `'top'` otherwise,
   matching the original.

The original script also **truncates** long `Value` cells (`abstract` /
`description_of_drawings`) and `Image_Path` cells for on-screen compactness.
That's a real data loss if baked into the file handed to `02b_postprocessing`,
so it's kept but off by default (`truncate_long_text=False`).


In [2]:
def format_review_workbook(
    df: pd.DataFrame,
    output_xlsx: Path,
    truncate_long_text: bool = False,
    min_col_width: int = 8,
    max_col_width: int = 60,
) -> None:
    """Write df to output_xlsx as TWO sheets:

    - "Review"  — flat, machine-readable, NO cell merging: what
                  02b_postprocessing / proc.load_review_images consume.
                  (Merged cells read back as NaN in pandas — merging the data
                  sheet silently destroys Patent_ID/Value on round-trip.)
    - "Compact" — the human view per src/Rearranging th eexcell.py:
                  consecutive identical cells merged, wrap text, auto-fit.

    Styling uses shared Alignment/Font/Fill objects — constructing one per
    cell makes openpyxl take minutes on a 20k-row batch.
    """
    headers = list(df.columns)
    truncate_fields = {"abstract", "description_of_drawings"}
    header_fill = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")
    header_font = Font(bold=True)
    align_top = Alignment(vertical="top", wrap_text=True)
    align_center = Alignment(vertical="center", wrap_text=True)

    def _cell_value(row, col_name):
        val = row[col_name]
        if pd.isna(val):
            return None
        val_str = str(val)
        if truncate_long_text:
            if col_name == "Value" and str(row.get("Field")) in truncate_fields and len(val_str) > 25:
                val_str = val_str[:25] + "..."
            if col_name == "Image_Path" and len(val_str) > 15:
                val_str = val_str[:15] + "..."
        return val_str

    def _fill_sheet(ws, merge: bool):
        ws.append(headers)
        for col_num in range(1, len(headers) + 1):
            cell = ws.cell(row=1, column=col_num)
            cell.fill = header_fill
            cell.font = header_font

        for _, row in df.iterrows():
            ws.append([_cell_value(row, c) for c in headers])

        # Wrap-text/top-align every data cell up front (shared object, cheap);
        # merge anchors get re-set to center below.
        for row_cells in ws.iter_rows(min_row=2):
            for cell in row_cells:
                cell.alignment = align_top

        if merge:
            # Merge consecutive rows with identical values, per column.
            for col in range(1, ws.max_column + 1):
                start_row = 2
                for row in range(3, ws.max_row + 1):
                    val_prev = ws.cell(row=row - 1, column=col).value
                    val_curr = ws.cell(row=row, column=col).value
                    if val_curr != val_prev or val_prev is None:
                        if (row - 1) > start_row and val_prev is not None:
                            ws.merge_cells(start_row=start_row, start_column=col, end_row=row - 1, end_column=col)
                            ws.cell(row=start_row, column=col).alignment = align_center
                        start_row = row
                if ws.max_row > start_row and ws.cell(row=start_row, column=col).value is not None:
                    ws.merge_cells(start_row=start_row, start_column=col, end_row=ws.max_row, end_column=col)
                    ws.cell(row=start_row, column=col).alignment = align_center

        # Auto-fit column widths from actual content (the original script's
        # hardcoded {'A': 18, ...} assumed one fixed 7-column layout).
        for col_num, col_name in enumerate(headers, start=1):
            col_letter = get_column_letter(col_num)
            longest = max(
                [len(col_name)] + [len(str(v)) for v in df[col_name].dropna().astype(str)],
                default=len(col_name),
            )
            ws.column_dimensions[col_letter].width = max(min_col_width, min(longest + 2, max_col_width))

    wb = openpyxl.Workbook()
    ws_data = wb.active
    ws_data.title = "Review"
    _fill_sheet(ws_data, merge=False)
    _fill_sheet(wb.create_sheet("Compact"), merge=True)

    wb.save(output_xlsx)
    print(f"Successfully generated formatted workbook (Review + Compact): {output_xlsx}")


## Section 3 — Data Loading

Read the raw `reviewed_patents_<batch>.xlsx` export from the review stage.
Never mutate/overwrite the original export file — all cleaning happens on an
in-memory copy, written out as a new file in Section 6.


In [3]:
# ── Input: the CORRECTED working copy, not the frozen human export ──────────
# 03_HUMAN_wizard_exports/ is frozen (chmod 444 + PRISTINE_MANIFEST.sha256, see
# scripts/freeze_exports.py): it is the record of what the annotator clicked and is
# never written to again. Every correction after export — cross-batch duplicate
# links, codebook conformance, physics-consistency fixes — is applied to a copy in
# corrected_wizard_exports/, and THAT is what this stage consumes. A batch with no
# copy yet falls back to the frozen export, with a notice, so nothing silently stalls.
FROZEN_DIR  = Path(cfg["paths"]["html_review_exports"])
EXPORTS_DIR = Path(cfg["paths"]["corrected_wizard_exports"])
if not (EXPORTS_DIR / f"reviewed_patents_{sheet_name}.xlsx").exists():
    if list(EXPORTS_DIR.glob(f"reviewed_patents_{sheet_name}*.xlsx")):
        pass                      # a near-miss copy exists; the rescue below finds it
    else:
        print(f"⚠ no corrected working copy for {sheet_name} — reading the FROZEN human "
              f"export directly. Corrections belong in {EXPORTS_DIR.name}; create the copy with\n"
              f"    python scripts/freeze_exports.py --copy {sheet_name}")
        EXPORTS_DIR = FROZEN_DIR
REVIEWED_XLSX = EXPORTS_DIR / f"reviewed_patents_{sheet_name}.xlsx"

if not REVIEWED_XLSX.exists():
    # Friendly failure: show what IS there, and rescue near-miss filenames
    # (e.g. "reviewed_patents_Batch_05 .xlsx" with a stray space — a real case).
    available = sorted(EXPORTS_DIR.glob("reviewed_patents_*.xlsx"))
    near_miss = [p for p in available
                 if p.name.replace(" ", "") == REVIEWED_XLSX.name.replace(" ", "")]
    if near_miss:
        REVIEWED_XLSX = near_miss[0]
        print(f"⚠ exact filename not found — using near-match {REVIEWED_XLSX.name!r} "
              f"(consider renaming it to remove stray spaces)")
    else:
        batches = [p.stem.removeprefix("reviewed_patents_").strip() for p in available]
        raise FileNotFoundError(
            f"{REVIEWED_XLSX} not found.\n"
            f"  Batches with a wizard export in {EXPORTS_DIR}:\n    "
            + ("\n    ".join(sorted(set(batches))) if batches else "(none)")
            + f"\n  Either set sheet_name (Section 1) to one of those, or export "
            f"{sheet_name} from the wizard ('Export batch' button) first."
        )

# Long format: one row per (Patent_ID, Section, Sub_Dimension, Field, Value,
# Source, Image_Path). NOTE the wizard export is NOT the excel_schema.py
# source format: Values are "ID — Label" composites (use _strip_label), M3
# fields are card-prefixed (wing1_propKin, boom_t1_propKin, ...), and the
# edge-case tags live in META/t1EdgeTags.
xls = pd.ExcelFile(REVIEWED_XLSX)
if "Review" not in xls.sheet_names:
    raise ValueError(
        f"{REVIEWED_XLSX.name} has no 'Review' sheet (found: {xls.sheet_names}). "
        f"The wizard's 'Export batch' writes a flat 'Review' sheet — this file "
        f"looks like something else. Re-export from the wizard."
    )
df_raw = pd.read_excel(xls, sheet_name="Review")

_REQUIRED_COLS = ["Patent_ID", "Section", "Sub_Dimension", "Field", "Value",
                  "Source", "Image_Path"]
_missing_cols = [c for c in _REQUIRED_COLS if c not in df_raw.columns]
if _missing_cols:
    raise ValueError(
        f"{REVIEWED_XLSX.name} 'Review' sheet is missing required column(s) "
        f"{_missing_cols} (has: {list(df_raw.columns)}). This is not a wizard "
        f"batch export — re-export from the wizard."
    )

df = df_raw.copy()   # all downstream work happens on this copy

if SMOKE_N_PATENTS:
    _keep_ids = list(dict.fromkeys(df["Patent_ID"].astype(str)))[:SMOKE_N_PATENTS]
    df = df[df["Patent_ID"].astype(str).isin(_keep_ids)].reset_index(drop=True)
    print(f"SMOKE mode: subset to first {len(_keep_ids)} patent(s), {len(df)} rows")

n_ids = df["Patent_ID"].nunique()
_base_ids = df["Patent_ID"].astype(str).str.split("_arch").str[0]
n_base_patents = _base_ids.nunique()
n_arch_variants = (df["Patent_ID"].astype(str).str.contains("_arch")).sum() and \
    df.loc[df["Patent_ID"].astype(str).str.contains("_arch"), "Patent_ID"].nunique()
print(f"loaded {len(df)} rows / {n_ids} Patent_ID key(s) from {REVIEWED_XLSX.name}")
if n_arch_variants:
    print(f"  ({n_base_patents} distinct patent(s); {n_arch_variants} are "
          f"_archN multi-architecture variant rows on top of their base patent)")

# ── Resolve blank Image_Path cells in-memory (scripts/resolve_image_paths.py) ──
# The wizard's "📎 Attach/Replace Image" can only record a filename, never a
# disk path, so those rows come back with a blank Image_Path. The script
# fixes the file in place; here we apply the same find_image() lookup to the
# in-memory copy instead, so the raw export stays untouched and Section 6's
# output carries resolved paths.
#
# "Image: (fig N)" rows are text-referenced figures with no matched image on
# disk — placeholders by design, skipped. Unresolved files only WARN when
# their block is APPROVED: disapproved blocks (e.g. abandoned clipboard
# pastes like FR3143549A1's "Pasted image*.png") are dropped by Section 6's
# approved-only export anyway, so a missing file there is harmless.
from scripts.resolve_image_paths import find_image

matched_root = Path(cfg["paths"]["matched"])
img_names = df["Sub_Dimension"].astype(str).str.removeprefix("Image: ").str.strip()
is_placeholder = img_names.str.match(r"^\(fig .*\)$") | (img_names == "(none available)")
img_mask = (
    (df["Section"] == "T2")
    & df["Sub_Dimension"].astype(str).str.startswith("Image: ")
    & ~is_placeholder
    & (df["Image_Path"].isna() | (df["Image_Path"].astype(str).str.strip() == ""))
)

# block (Patent_ID, Sub_Dimension) -> approved?
status_rows = df[(df["Section"] == "T2") & (df["Field"] == "status")]
block_approved = {
    (str(r["Patent_ID"]), str(r["Sub_Dimension"])): str(r["Value"]).strip().lower() == "approved"
    for _, r in status_rows.iterrows()
}

resolved, unresolved_approved, unresolved_ignored = 0, [], 0
for idx, row in df[img_mask].iterrows():
    fname = img_names[idx]
    hit = find_image(matched_root, str(row["Patent_ID"]).strip(), fname)
    if hit:
        df.at[idx, "Image_Path"] = str(hit)
        resolved += 1
    elif block_approved.get((str(row["Patent_ID"]), str(row["Sub_Dimension"]))):
        unresolved_approved.append((row["Patent_ID"], fname))
    else:
        unresolved_ignored += 1

print(f"Image_Path resolution: {int(img_mask.sum())} blank (placeholders excluded) | "
      f"{resolved} resolved | {len(unresolved_approved)} unresolved APPROVED | "
      f"{unresolved_ignored} unresolved disapproved (harmless, dropped at export)")
if unresolved_approved:
    print("  ⚠ APPROVED images missing on disk — copy these into the patent's matched/ folder, re-run this cell:")
    for pid, fname in sorted(set(unresolved_approved)):
        print(f"    - {pid}: {fname}")


loaded 25584 rows / 393 Patent_ID key(s) from reviewed_patents_Batch_01.xlsx
  (352 distinct patent(s); 41 are _archN multi-architecture variant rows on top of their base patent)
Image_Path resolution: 45 blank (placeholders excluded) | 39 resolved | 0 unresolved APPROVED | 6 unresolved disapproved (harmless, dropped at export)


### A4 — Assert zero `duplicateType == '4'`

Guard, runs immediately after loading (before any Section 4 rule touches
`duplicateType`). The annotator's workflow expects every duplicate to have
been triaged into D1/D2/D3 — a `duplicateType == '4'` row means something
was left in the retired bucket and needs manual re-assignment before this
notebook processes the batch. Report-only in the sense that it changes
nothing — but unlike the other report-only checks, a nonzero count is fatal:
it stops the notebook rather than just flagging for later.


In [ ]:
# A4 — hard stop if any duplicateType == "4" survived triage.
# Self-contained id-strip (Value split on " — ") since this runs before
# Section 4 defines _strip_label — deliberately, so this guard fires as
# early as possible, right after loading.
_dt4_mask = (
    (df["Field"] == "duplicateType")
    & df["Value"].notna()
    & (df["Value"].astype(str).str.split(" — ").str[0].str.strip() == "4")
)
_dt4_ids = sorted(df.loc[_dt4_mask, "Patent_ID"].astype(str).unique())

if _dt4_ids:
    print(f"A4 ({datetime.now().date().isoformat()}): {len(_dt4_ids)} patent(s) still carry "
          f"duplicateType == '4' — STOPPING. Re-assign these to D1/D2/D3 before re-running:")
    for pid in _dt4_ids:
        print(f"  - {pid}")
    raise AssertionError(
        f"A4: {len(_dt4_ids)} patent(s) with duplicateType == '4' (see printed list above)."
    )
else:
    print(f"A4 ({datetime.now().date().isoformat()}): 0 patent(s) with duplicateType == '4' — OK.")


## Section 4 — Validation Rule Logic

In [4]:
def _strip_label(value):
    """Wizard export Values are "ID — Label" composites (withLabel format,
    e.g. "TP — Vectored Thrust — Tilt Propulsors"). Return just the ID part;
    None for NaN/blank. Mirrors the HTML's stripLabel().
    """
    if pd.isna(value):
        return None
    return str(value).split(" — ")[0].strip()


def _append_flag(df: pd.DataFrame, patent_ids, message: str) -> None:
    """In-place: append `message` to pre_process_flags for every row whose
    Patent_ID is in patent_ids (idempotent — skips patents that already
    carry that exact message).
    """
    if "pre_process_flags" not in df.columns:
        df["pre_process_flags"] = ""
    df["pre_process_flags"] = df["pre_process_flags"].fillna("")

    mask = df["Patent_ID"].isin(patent_ids)
    df.loc[mask, "pre_process_flags"] = df.loc[mask, "pre_process_flags"].apply(
        lambda existing: existing if message in existing else f"{existing}{message}"
    )


### Rule A — Combined Thrust (flag for review, target = CVT)

"Combined Thrust" **is** the existing G1 option `CVT — Vectored Thrust —
Combined` — no new taxonomy id needed.

For each `Patent_ID`, compare its stripped `topType` (Section `G1`) against
its per-card kinematics rows (`Field` ends with `_propKin`: `wing1_propKin`,
`fuselage_propKin`, `boom_t1_propKin`, ...; stripped ids
`Fixed | Tilt | Vectored | Cyclic`):

- `topType == TP` **and** propKin mixes `Fixed` + tilting → the patent looks
  like combined thrust mislabelled as pure tilt-propulsor. **Flag only**
  (`"Review: Potential Combined Thrust (TP→CVT?);"`) — the Section 5 UI shows
  the image and the reviewer decides via **Update to CVT** / **Keep As Is**.
- `topType == CVT` with mixed propKin is *consistent* — no flag.

The rule deliberately does not auto-overwrite: since the original labels may
simply be wrong, the human pass in Section 5 is the decision point.


In [5]:
CVT_VALUE = "CVT — Vectored Thrust — Combined"   # the wizard's own composite for CVT


def _flag_combined_thrust(df: pd.DataFrame) -> pd.DataFrame:
    """Rule A — Combined Thrust candidates (flag-only; human decides in the
    Section 5 UI whether to switch topType TP → CVT).

    Flags patents whose stripped topType is TP while their *_propKin rows mix
    "Fixed" with a tilting mechanism (Tilt/Vectored/Cyclic). CVT patents with
    mixed propKin are already consistent and are left alone.
    """
    df = df.copy()

    top_type_by_patent = (
        df.loc[df["Field"] == "topType"]
        .set_index("Patent_ID")["Value"].map(_strip_label)
    )

    propkin_rows = df.loc[df["Field"].astype(str).str.endswith("_propKin")]

    def _is_mixed(values: pd.Series) -> bool:
        vals = {_strip_label(v) for v in values} - {None}
        return "Fixed" in vals and bool(vals - {"Fixed"})

    mixed_by_patent = propkin_rows.groupby("Patent_ID")["Value"].apply(_is_mixed)

    flagged_ids = [
        pid for pid, is_mixed in mixed_by_patent.items()
        if is_mixed and top_type_by_patent.get(pid) == "TP"
    ]

    if flagged_ids:
        _append_flag(df, flagged_ids, "Review: Potential Combined Thrust (TP→CVT?);")

    return df


In [7]:
# RETIRED (2026-07-22, R5-02 conflict fix) — removed from PIPELINE_RULES,
# kept here unused for history. This rule force-cleared empTilts=True -> False
# whenever every *_propKin row stripped to "Fixed" — but propKinLock() makes
# that condition true BY DESIGN for topType in {TW, TB, DS, SLC, SRW, MR}, so
# this silently erased legitimate empTilts=True answers on exactly the
# architectures (TW, TB) that codebook R5-02 says must be allowed to have it,
# with no reviewer ever told. Superseded by the report-only S3 soft check
# (_check_emptilts_on_rigid_arch, below) — flags for human review instead of
# overwriting. See the retroactive dry-run check directly below this cell.
def _flag_fixed_empennage(df: pd.DataFrame) -> pd.DataFrame:
    """Rule B — Fixed Empennage.

    Patents with no tilting propulsor (all *_propKin rows strip to "Fixed")
    get any empTilts=True row forced to False, and are flagged. Patents with
    no empTilts row at all are untouched (already correct).
    """
    df = df.copy()

    propkin_rows = df.loc[df["Field"].astype(str).str.endswith("_propKin")]
    has_tilt_by_patent = propkin_rows.groupby("Patent_ID")["Value"].apply(
        lambda values: any(_strip_label(v) not in (None, "Fixed") for v in values)
    )
    zero_tilt_ids = has_tilt_by_patent[~has_tilt_by_patent].index

    emp_tilts_mask = (
        (df["Field"] == "empTilts")
        & df["Patent_ID"].isin(zero_tilt_ids)
        & (df["Value"] == True)  # noqa: E712 — explicit bool compare, not truthiness
    )
    forced_ids = df.loc[emp_tilts_mask, "Patent_ID"].unique().tolist()

    if forced_ids:
        df.loc[emp_tilts_mask, "Value"] = False
        _append_flag(df, forced_ids, "Empennage tilt forced to False;")

    return df


### Rule C (Bug-2 fix) — Duplicate Chain Tagging, D3 excluded as a propagation destination

The duplicate link lives in the T1 fields `isDuplicate` (bool) +
`duplicateId` (Value = the plain Patent_ID it duplicates — no label suffix).
Chains (A dup-of B dup-of C) are resolved as connected components of an
undirected graph built from those `duplicateId` edges via `networkx`.

The "UAV, but similar enough" tag (`UAVSimilar`) is an **edge-case tag**, not
a disapproval reason: it lives in the `META` section, Field `t1EdgeTags`,
Sub_Dimension `"Review Metadata"`. If any patent in a connected component
carries `UAVSimilar` there, every OTHER patent in that component gets it too
(appended comma-separated to an existing t1EdgeTags row, or a new row is
created) and is flagged with `"UAV Tag Propagated;"`.

**Bug-2 fix:** propagation used to ignore `duplicateType` entirely — a D3
member (`duplicateType == "3"`, "same plane/small changes" — a MODIFIED
variant, not the same aircraft) would silently inherit `UAVSimilar` from
elsewhere in its chain even though D3 is supposed to be an independently-
decided tag. Now, before propagating, any node whose OWN `duplicateType` is
`"3"` is excluded from the propagation DESTINATION set — a D3 patent can
still be the SOURCE of the tag (if it carries `UAVSimilar` itself, that
stays and still propagates outward to non-D3 members), it just never
receives the tag automatically from the rest of its chain.


In [ ]:
def _propagate_duplicate_tag(df: pd.DataFrame) -> pd.DataFrame:
    """Rule C (Bug-2 fix) — Duplicate Chain Tagging.

    Builds an undirected graph of Patent_ID <-> duplicateId edges, finds
    connected components (chains), and — if any patent in a component carries
    "UAVSimilar" in its META/t1EdgeTags row — stamps that tag onto every
    OTHER patent in the component EXCEPT those whose own duplicateType is
    "3" (D3 — modified variant): a D3 patent may still be a tag SOURCE, but
    is never an automatic propagation DESTINATION (its own tag decision is
    independent of the rest of the chain).
    """
    df = df.copy()

    graph = nx.Graph()
    graph.add_nodes_from(df["Patent_ID"].astype(str).unique())

    dup_edges = df.loc[
        (df["Field"] == "duplicateId") & df["Value"].notna() & (df["Value"].astype(str).str.strip() != "")
    ]
    for _, row in dup_edges.iterrows():
        graph.add_edge(str(row["Patent_ID"]), str(row["Value"]).strip())

    # "UAV, but similar enough" lives in the META section's t1EdgeTags field
    # (Value contains the tag id "UAVSimilar") — NOT in t1DisapproveReason.
    edge_tag_rows = df["Field"] == "t1EdgeTags"
    tagged_ids = set(
        df.loc[edge_tag_rows & df["Value"].astype(str).str.contains("UAVSimilar", na=False), "Patent_ID"].astype(str)
    )

    # Own duplicateType per patent — D3 ("3") nodes are excluded from the
    # propagation DESTINATION set (Bug-2 fix). A D3 patent's OWN tag (if it
    # has one) is untouched and still counts as a valid source for others.
    dup_type_rows = df.loc[df["Field"] == "duplicateType"]
    dup_type_by_patent = dup_type_rows.set_index(dup_type_rows["Patent_ID"].astype(str))["Value"].map(_strip_label)
    d3_ids = set(dup_type_by_patent[dup_type_by_patent == "3"].index)

    # Any chain (component of size > 1) that contains a tagged patent needs
    # the tag propagated to its other members — except D3 destinations.
    propagate_ids, d3_excluded_ids = set(), set()
    for component in nx.connected_components(graph):
        if len(component) > 1 and (component & tagged_ids):
            candidates = component - tagged_ids
            propagate_ids |= (candidates - d3_ids)
            d3_excluded_ids |= (candidates & d3_ids)

    if propagate_ids:
        in_chain = df["Patent_ID"].astype(str).isin(propagate_ids)

        # Existing t1EdgeTags rows: append the tag (comma-separated) unless present.
        existing_mask = edge_tag_rows & in_chain
        df.loc[existing_mask, "Value"] = df.loc[existing_mask, "Value"].apply(
            lambda v: "UAVSimilar" if pd.isna(v) or not str(v).strip()
            else (str(v) if "UAVSimilar" in str(v) else f"{v},UAVSimilar")
        )

        # Patents in the chain with no t1EdgeTags row yet — add one, matching
        # the wizard export's 7-column schema.
        already_updated = set(df.loc[existing_mask, "Patent_ID"].astype(str))
        missing_ids = propagate_ids - already_updated
        if missing_ids:
            new_rows = pd.DataFrame([
                {
                    "Patent_ID": pid, "Section": "META", "Sub_Dimension": "Review Metadata",
                    "Field": "t1EdgeTags", "Value": "UAVSimilar",
                    "Source": "rule_c_propagation", "Image_Path": None, "pre_process_flags": "",
                }
                for pid in missing_ids
            ])
            df = pd.concat([df, new_rows], ignore_index=True)

        _append_flag(df, propagate_ids, "UAV Tag Propagated;")

    print(f"Rule C (Bug-2 fix, {datetime.now().date().isoformat()}): {len(propagate_ids)} patent(s) "
          f"tagged UAVSimilar by propagation | {len(d3_excluded_ids)} D3 patent(s) excluded from "
          f"propagation (own duplicateType == '3' — spot-check if nonzero): "
          f"{sorted(d3_excluded_ids) if d3_excluded_ids else '[]'}")
    return df


### Rule D — Duplicate Inheritance (topType + isApproved from chain root)

The HTML wizard auto-copies M1–M3 to duplicates but **not G1**, and doesn't
force an approval choice on them. Verified on Batch_01: ~131 approved patents
missing `topType` and all 32 NaN-`isApproved` patents are duplicates whose
chain root carries the value.

For every member of a duplicate chain (per Rule C's graph, following
`duplicateId` links to the chain root):

- missing/blank `topType` → copy the root's `topType` (new G1 row,
  `Source = "rule_d_inherited"`), flag `"G1 inherited from duplicate root;"`.
- `isApproved` NaN → copy the root's `isApproved`, flag
  `"Approval inherited from duplicate root;"`.

Members whose root also lacks the value, and non-duplicates missing topType,
are flagged `"Missing topType — needs manual review;"` for the Section 5 UI
instead (nothing to inherit).

The 41 `_archN`-suffixed ids lacking `isApproved` rows are left alone —
approval lives on the base patent by design.


In [9]:
def _resolve_chain_root(patent_id: str, dup_target: pd.Series) -> str:
    """Follow duplicateId links to the chain root (cycle-safe)."""
    seen = set()
    current = str(patent_id)
    while current not in seen:
        seen.add(current)
        target = dup_target.get(current)
        if target is None or pd.isna(target) or not str(target).strip():
            return current
        current = str(target).strip()
    return current  # cycle — return where we stopped


def _inherit_from_duplicate_root(df: pd.DataFrame) -> pd.DataFrame:
    """Rule D — duplicates inherit topType (G1) and isApproved from their
    chain root when they lack a value of their own.

    "Missing topType" is only flagged where it's actually unexpected:
    disapproved patents skip G1 by design, multi-arch patents carry topType
    on their _archN ids (not the base id), and exact duplicates (duplicateType
    "2" — "Images AND aircraft the same") skip G1 by design too: the wizard
    never writes them a topType row, and Section 6's export enforces that they
    stay label-less (the pipeline resolves their full labels via duplicateId).
    """
    df = df.copy()

    dup_rows = df.loc[(df["Field"] == "duplicateId") & df["Value"].notna()
                      & (df["Value"].astype(str).str.strip() != "")]
    dup_target = dup_rows.set_index(dup_rows["Patent_ID"].astype(str))["Value"]
    is_dup_ids = set(dup_target.index)

    top_type_rows = df.loc[df["Field"] == "topType"]
    top_type_by_patent = top_type_rows.set_index(top_type_rows["Patent_ID"].astype(str))["Value"]
    appr_rows = df.loc[df["Field"] == "isApproved"]
    appr_by_patent = appr_rows.set_index(appr_rows["Patent_ID"].astype(str))["Value"]

    # Exact duplicates (type "2") carry no G1 of their own by design — never
    # inherit/flag topType for them, even though they're technically "missing" it.
    dup_type_rows = df.loc[df["Field"] == "duplicateType"]
    dup_type_by_patent = dup_type_rows.set_index(dup_type_rows["Patent_ID"].astype(str))["Value"].map(_strip_label)
    exact_dup_ids = set(dup_type_by_patent[dup_type_by_patent == "2"].index)

    all_id_strings = df["Patent_ID"].astype(str).unique()
    # arch-suffixed ids never carry their own isApproved (approval lives on
    # the base patent), and multi-arch BASE ids never carry their own topType
    # (it lives on the _archN ids) — both excluded from "missing" checks.
    arch_bases = {p.split("_arch")[0] for p in all_id_strings if "_arch" in p}
    all_ids = {p for p in all_id_strings if "_arch" not in p}

    def _is_disapproved(pid):
        return str(appr_by_patent.get(pid)).strip().lower() == "false"

    new_rows, tt_inherited, appr_inherited, needs_manual = [], [], [], []
    for pid in sorted(all_ids):
        needs_tt = (pd.isna(top_type_by_patent.get(pid))
                    and pid not in arch_bases and not _is_disapproved(pid)
                    and pid not in exact_dup_ids)
        has_appr = pd.notna(appr_by_patent.get(pid))
        if not needs_tt and has_appr:
            continue

        if pid in is_dup_ids:
            root = _resolve_chain_root(pid, dup_target)
            root_tt = top_type_by_patent.get(root)
            root_appr = appr_by_patent.get(root)

            if needs_tt and pd.notna(root_tt):
                new_rows.append({
                    "Patent_ID": pid, "Section": "G1", "Sub_Dimension": "Topology Type",
                    "Field": "topType", "Value": root_tt,
                    "Source": "rule_d_inherited", "Image_Path": None, "pre_process_flags": "",
                })
                tt_inherited.append(pid)
            elif needs_tt:
                needs_manual.append(pid)

            if not has_appr and pd.notna(root_appr):
                appr_mask = (df["Field"] == "isApproved") & (df["Patent_ID"].astype(str) == pid)
                if appr_mask.any():
                    df.loc[appr_mask, "Value"] = root_appr
                else:
                    new_rows.append({
                        "Patent_ID": pid, "Section": "T1", "Sub_Dimension": "T1 — Approval Status",
                        "Field": "isApproved", "Value": root_appr,
                        "Source": "rule_d_inherited", "Image_Path": None, "pre_process_flags": "",
                    })
                appr_inherited.append(pid)
        elif needs_tt:
            # Non-duplicate, approved, single-arch, no topType — needs eyes.
            needs_manual.append(pid)

    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    if tt_inherited:
        _append_flag(df, tt_inherited, "G1 inherited from duplicate root;")
    if appr_inherited:
        _append_flag(df, appr_inherited, "Approval inherited from duplicate root;")
    if needs_manual:
        _append_flag(df, needs_manual, "Missing topType — needs manual review;")

    print(f"Rule D: topType inherited {len(tt_inherited)} | isApproved inherited "
          f"{len(appr_inherited)} | needs manual review {len(needs_manual)}")
    return df


### A1 (Bug-1 fix, part 1) — Fixed-Architecture `acState` Auto-Set

Per current codebook (R2-03a): a FIXED-architecture patent (`topType ∈
{SLC, SRW, RC, MR, HB, PFV}`) has no meaningful per-figure flight state —
every figure gets `acState` forced to `HoverCruise`, **including** any figure
currently marked `Ground` (the old Rule E below used to remap those to
`Other`, which is wrong for fixed architectures — see the bug note in Rule
E's own cell). Only CONVERTIBLE architectures (`TW, TP, DS, CVT, TB, PTC`)
keep a real per-figure flight state and go through Rule E instead.

Runs BEFORE Rule E in the pipeline so fixed-arch figures never reach Rule
E's Ground→Other remap at all. The prior value (whatever it was —
Ground, Hover, Cruise, ...) is preserved as a sibling `acState_legacy` row,
never overwritten in place, so nothing is silently lost if this needs
auditing later.


In [ ]:
FIXED_ARCH_TYPES = {"SLC", "SRW", "RC", "MR", "HB", "PFV"}
CONVERTIBLE_ARCH_TYPES = {"TW", "TP", "DS", "CVT", "TB", "PTC"}

# v15.2 renamed this option's DISPLAY label to "Invariant" (the id "HoverCruise"
# is unchanged). The wizard now exports "HoverCruise — Invariant", so writing the
# old composite back left one file carrying two labels for the same id. Values are
# always read through _strip_label(), so only the label text needed correcting.
HOVERCRUISE_VALUE = "HoverCruise — Invariant"


def _force_hovercruise_fixed_arch(df: pd.DataFrame) -> pd.DataFrame:
    """A1 (Bug-1 fix, part 1) — force acState=HoverCruise on every figure of
    a FIXED-architecture patent (topType in FIXED_ARCH_TYPES), regardless of
    its current value. The old value is kept as a sibling acState_legacy row
    (never lost). Rows already HoverCruise are left untouched (idempotent —
    no spurious legacy row for a no-op).
    """
    df = df.copy()

    top_type_by_patent = (
        df.loc[df["Field"] == "topType"].set_index("Patent_ID")["Value"].map(_strip_label)
    )
    fixed_ids = {pid for pid, tt in top_type_by_patent.items() if tt in FIXED_ARCH_TYPES}

    mask = (
        (df["Field"] == "acState")
        & df["Patent_ID"].isin(fixed_ids)
        & (df["Value"].map(_strip_label) != "HoverCruise")
    )
    changed = df.loc[mask].copy()   # captures OLD values before overwrite

    if not changed.empty:
        breakdown = changed["Value"].map(_strip_label).fillna("(blank)").value_counts().to_dict()

        legacy_rows = changed.copy()
        legacy_rows["Field"] = "acState_legacy"
        legacy_rows["Source"] = "bug1_fixed_arch_legacy"

        df.loc[mask, "Value"] = HOVERCRUISE_VALUE   # apply BEFORE concat (same index)
        df = pd.concat([df, legacy_rows], ignore_index=True)

        affected_ids = changed["Patent_ID"].unique().tolist()
        _append_flag(df, affected_ids, "acState forced to HoverCruise (fixed arch);")
    else:
        breakdown, affected_ids = {}, []

    print(f"A1 (Bug-1 fix, {datetime.now().date().isoformat()}): {len(changed)} acState row(s) "
          f"forced to HoverCruise on {len(affected_ids)} fixed-architecture patent(s) "
          f"| previous-value breakdown: {breakdown}")
    return df


### Rule E (narrowed, Bug-1 fix part 2) — Ground → Other, CONVERTIBLE architectures only

`acState` (Section T2, per figure) has 6 wizard options: `Ground, Hover,
Transition, Cruise, Unclear, NonApplicable`. Verified on Batch_01: `Ground`
is the most ambiguous in practice (a "ground" drawing is often just an
uncommitted static illustration, not a meaningful flight-state observation),
so it's folded into `"Other — Other"`, leaving `Hover`/`Cruise`/`Transition`
as the three real flight states plus `Unclear`/`NonApplicable`/`Other`.

**Bug-1 fix:** this used to run unconditionally, over-writing `Ground` to
`Other` even on FIXED architectures — wrong, since A1 above already forces
ALL of a fixed-architecture patent's figures to `HoverCruise`. This rule now
only touches patents whose `topType` is a CONVERTIBLE architecture
(`TW, TP, DS, CVT, TB, PTC` — `CONVERTIBLE_ARCH_TYPES`, defined in A1's cell
above). Fixed-architecture patents never reach this rule at all (A1 already
overwrote every one of their `acState` rows to `HoverCruise` earlier in the
pipeline, so the mask below naturally excludes them).

Every `acState` row whose stripped value is `"Ground"`, on a CONVERTIBLE-
architecture patent, is rewritten to `"Other — Other"` and its patent
flagged `"acState Ground → Other;"`. `Other` was added as a real option in
the HTML wizard (`UI_for_taxonomy_caracterization_13_0.html`, `AC_STATE`
array) so the file still round-trips if reopened there.


In [ ]:
def _remap_ground_to_other(df: pd.DataFrame) -> pd.DataFrame:
    """Rule E (narrowed, Bug-1 fix part 2) — fold per-figure acState "Ground"
    into "Other", restricted to CONVERTIBLE-architecture patents only (fixed
    architectures are handled exclusively by A1, above)."""
    df = df.copy()

    top_type_by_patent = (
        df.loc[df["Field"] == "topType"].set_index("Patent_ID")["Value"].map(_strip_label)
    )
    convertible_ids = {pid for pid, tt in top_type_by_patent.items() if tt in CONVERTIBLE_ARCH_TYPES}

    mask = (
        (df["Field"] == "acState")
        & df["Patent_ID"].isin(convertible_ids)
        & (df["Value"].map(_strip_label) == "Ground")
    )
    affected_ids = df.loc[mask, "Patent_ID"].unique().tolist()

    if affected_ids:
        df.loc[mask, "Value"] = "Other — Other"
        _append_flag(df, affected_ids, "acState Ground → Other;")

    print(f"Rule E ({datetime.now().date().isoformat()}, convertible-arch only): "
          f"{int(mask.sum())} acState row(s) remapped Ground → Other "
          f"({len(affected_ids)} patent(s))")
    return df


### New (Part C) — Multi-Architecture Aircraft Name Suffix

When a patent has more than one architecture (`_archN`-suffixed
`Patent_ID`s — `archCount > 1`), every architecture shares the exact same
`aircraftName` (the HTML wizard writes `aircraftName` once, on the base
patent id, at T1 — never per-arch; see `recordToRows()` /
`UI_for_taxonomy_caracterization_14_0.html`). That's the reported bug: two
visually and kinematically distinct architectures under one patent read as
having the identical name.

Fix: for each base patent with `_archN` ids, append a distinguishing letter
suffix to each architecture's own copy of the name (arch 1 → "A", arch 2 →
"B", ...) — e.g. base name "Bettor" becomes "Bettor A", "Bettor B", ... —
written as a **new** `aircraftName_perArch` row on each `_archN` id. The
base patent's own unsuffixed `aircraftName` row is left untouched (stays the
shared root name) — this only ADDS a field, never overwrites the original.
Patents that already had a blank/missing `aircraftName` are skipped (nothing
meaningful to suffix).


In [ ]:
def _suffix_multiarch_aircraft_names(df: pd.DataFrame) -> pd.DataFrame:
    """New (Part C) — give each _archN id of a multi-architecture patent its
    own distinguishing aircraftName_perArch (base name + a letter suffix:
    A, B, C, ...), since the wizard only ever writes one shared aircraftName
    on the base patent id. Adds a NEW row per architecture; the base
    patent's own aircraftName row is untouched.
    """
    df = df.copy()

    all_ids = df["Patent_ID"].astype(str).unique()
    arch_ids_by_base: dict = {}
    for pid in all_ids:
        if "_arch" in pid:
            base, _, suffix = pid.partition("_arch")
            try:
                n = int(suffix)
            except ValueError:
                continue
            arch_ids_by_base.setdefault(base, []).append((n, pid))

    name_rows = df.loc[df["Field"] == "aircraftName"]
    name_by_base = name_rows.set_index(name_rows["Patent_ID"].astype(str))["Value"]

    new_rows, affected_bases = [], []
    for base, arch_list in arch_ids_by_base.items():
        base_name = name_by_base.get(base)
        if pd.isna(base_name) or not str(base_name).strip():
            continue  # nothing meaningful to suffix
        for n, pid in sorted(arch_list):
            letter = chr(ord("A") + n - 1) if 1 <= n <= 26 else str(n)
            new_rows.append({
                "Patent_ID": pid, "Section": "T1", "Sub_Dimension": "Aircraft / Prototype Name",
                "Field": "aircraftName_perArch", "Value": f"{base_name} {letter}",
                "Source": "partc_multiarch_name_suffix", "Image_Path": None, "pre_process_flags": "",
            })
        affected_bases.append(base)

    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
        _append_flag(df, affected_bases, "aircraftName_perArch suffix added (multi-architecture);")

    print(f"New/Part C ({datetime.now().date().isoformat()}): {len(affected_bases)} multi-architecture "
          f"patent(s) affected | {len(new_rows)} aircraftName_perArch row(s) added")
    return df


### S3 / D5 — Empennage-tilt rigid-architecture check & confidence-field audit

Addendum (2026-07-21), continued: two more follow-ups from the same wizard v14 session.
Both are **report-only** — run after the pipeline (need the confirmed, post-migration `df`,
same reason the Completeness Check runs there) — and never modify data.

**S3** (soft check): `empTilts=True` is unusual on a rigid/near-rigid architecture
(`topType ∈ {TB, TW}` — TW included per the annotator's explicit "include both, I'll eyeball
it" instruction, not because it's necessarily wrong). Flags for review; separately escalates
the subset with a **blank** justification note — the wizard's own `nextBlockers()` makes that
note mandatory whenever the checkbox is ticked, so a blank note here means either a pre-v14
save or something bypassed the wizard's own gate, which is a real data-quality signal, not
just an eyeball item.

> **Known limitation, inherited, not introduced here:** this reads `empTilts` as migrated by
> the former `Step 0b` (removed 2026-08-23; see the Orchestration cell), which had a
> documented gap — it never recognized legacy
> `empKin='Stabilator'` as tilting (see `AUDIT_CODEBOOK_V2_STATUS.md` / memory
> `02a-v2-compat-review`, conflict #1). Until that's fixed, S3 under-counts Stabilator
> patents reviewed before v14. Not silently glossed over — logged in the cell output below.

**D5** (integrity check): the original brief assumes a `Confidence` column exists at each
stage (T1/G1/M1/M2/M3) as dormant scaffolding. **Verified against the wizard's own
`reviewRow()`** (`UI_for_taxonomy_caracterization_14_0.html`) that it accepts a `confidence`
parameter but never writes it to the exported row — `reviewed_patents_<batch>.xlsx` (what this
notebook loads) genuinely has no such column, by construction, not as a bug. Rather than
silently fabricate a check on a column that isn't there, D5 below defensively looks for
**any** column or `Field` value matching "confidence" (covering both a wide-column and a
long-format-row interpretation) and reports honestly on what it actually finds — including
"nothing found," which is itself the answer if that's what the annotator wanted confirmed.


In [ ]:
def _check_emptilts_on_rigid_arch(df: pd.DataFrame):
    """S3 — soft, report-only: empTilts=True on topType in {TB, TW}.
    Returns (full_worklist_df, missing_justification_df).
    """
    top_type_by_patent = (
        df.loc[df["Field"] == "topType"].set_index("Patent_ID")["Value"].map(_strip_label)
    )
    tilts_rows = df.loc[(df["Field"] == "empTilts") & (df["Value"] == True)]  # noqa: E712
    notes_by_patent = df.loc[df["Field"] == "empTiltsNote"].set_index("Patent_ID")["Value"]

    flagged = []
    for pid in sorted(tilts_rows["Patent_ID"].astype(str).unique()):
        tt = top_type_by_patent.get(pid)
        if tt not in ("TB", "TW"):
            continue
        note = notes_by_patent.get(pid)
        has_note = pd.notna(note) and str(note).strip() != ""
        flagged.append({"Patent_ID": pid, "topType": tt,
                         "empTiltsNote": note if has_note else None,
                         "missing_justification": not has_note})

    worklist = pd.DataFrame(flagged, columns=["Patent_ID", "topType", "empTiltsNote", "missing_justification"])
    missing = worklist.loc[worklist["missing_justification"]].drop(columns=["missing_justification"]) if len(worklist) else \
        pd.DataFrame(columns=["Patent_ID", "topType", "empTiltsNote"])

    print(f"S3: {len(worklist)} patent(s) with empTilts=True on TB/TW (eyeball, not auto-fixed) | "
          f"{len(missing)} of those have NO justification note (missing_justification list — real data-quality issue)")
    return worklist, missing


def _check_confidence_constancy(df: pd.DataFrame) -> dict:
    """D5 — integrity check, report-only. See the markdown above for why this
    is written defensively (checks for the column's/field's actual existence
    rather than assuming the brief's premise holds for this file).
    """
    result = {"wide_columns": [], "field_rows": False, "per_stage": {}}
    stages = ["T1", "G1", "M1", "M2", "M3"]

    wide_cols = [c for c in df.columns if "confidence" in c.lower()]
    result["wide_columns"] = wide_cols
    for col in wide_cols:
        for stage in stages:
            vals = df.loc[df["Section"] == stage, col].dropna()
            if vals.empty:
                continue
            uniq = vals.unique()
            result["per_stage"][(col, stage)] = {
                "n_rows": len(vals), "n_unique": len(uniq), "constant": len(uniq) == 1,
                "value_or_top5": uniq[0] if len(uniq) == 1 else vals.value_counts().head(5).to_dict(),
                "example_patent_ids": df.loc[(df["Section"] == stage) & df[col].notna(),
                                              "Patent_ID"].astype(str).unique()[:5].tolist(),
            }

    field_rows = df.loc[df["Field"].astype(str).str.lower() == "confidence"]
    result["field_rows"] = not field_rows.empty
    if not field_rows.empty:
        for stage in stages:
            vals = field_rows.loc[field_rows["Section"] == stage, "Value"].dropna()
            if vals.empty:
                continue
            uniq = vals.unique()
            result["per_stage"][("Field==confidence", stage)] = {
                "n_rows": len(vals), "n_unique": len(uniq), "constant": len(uniq) == 1,
                "value_or_top5": uniq[0] if len(uniq) == 1 else vals.value_counts().head(5).to_dict(),
                "example_patent_ids": field_rows.loc[field_rows["Section"] == stage, "Patent_ID"].astype(str).unique()[:5].tolist(),
            }

    if not wide_cols and not result["field_rows"]:
        print("D5: no column or Field value matching 'confidence' found in this batch's "
              "loaded data. Verified: the wizard's reviewRow() never writes a Confidence "
              "column — reviewed_patents_<batch>.xlsx genuinely has none. This may describe "
              "ml_predict_labels_<batch>.xlsx (excel_schema.py's richer format) instead, "
              "which 02a does not load. Flagging for the annotator rather than guessing.")
    else:
        print(f"D5: found wide column(s) {wide_cols or '(none)'}"
              f"{' + Field==confidence rows' if result['field_rows'] else ''} — constancy per stage:")
        for (col, stage), info in result["per_stage"].items():
            status = "constant (dormant/default)" if info["constant"] else "NOT constant — investigate"
            print(f"  {col} @ {stage}: {status} | n={info['n_rows']} | examples={info['example_patent_ids']}")
    return result


### Completeness Check (report-only — "are all fields filled in?")

Runs last, after Rule D has inherited what it can from duplicate roots. For
every **approved** patent that is not an *exact* duplicate (`duplicateType
== "2"` — the only category that is legitimately label-less; types 1/3/4
carry their own G1-M3), checks that the required label rows exist and are
non-blank: `topType`, `wCount`, `empType`, `latSym`, `longSym`, plus at least
one `_propKin` row somewhere in M3 (every architecture has *some* propulsor
kinematics recorded). Multi-arch patents are checked per `_archN` id.

This doesn't fix anything — it flags gaps as `"Incomplete: <fields>;"` so
they surface in the Section 5 queue, and prints a summary so you can see at
a glance whether a batch is clean before exporting.


In [13]:
_OPTIONAL_FIELD_HINTS = ("note", "comment", "override", "uncertain", "othernote")


def _check_label_completeness(df: pd.DataFrame) -> pd.DataFrame:
    """Report-only: flag approved patents with any blank G1/M1/M2/M3 row
    (excluding free-text/optional fields — notes, quick overrides,
    uncertain-flags, "other" clarifications).

    Only duplicateType "2" ("Images AND aircraft the same" — a literal
    identical copy) is legitimately label-less by design (Section 6 strips
    its own T2/G1/M1-M3 and resolves everything via duplicateId lookup).
    duplicateType "1" (D1 — Same Aircraft, wizard auto-copies M1-M3+G1) and
    "3" (D3 — Same aircraft modified, copy is an editable starting point)
    both carry their OWN G1-M3 labels and must be checked like any
    non-duplicate patent — exempting every isDuplicate=True patent
    regardless of type was silently hiding genuinely incomplete type-1/3
    patents from review.

    Type "4" (Component Overlap) was RETIRED in wizard v14 and is no longer
    selectable; the A4 assert in Section 3 hard-stops if any survives. A
    v15.2 export can only contain D1/D2/D3.
    """
    df = df.copy()

    appr_rows = df.loc[df["Field"] == "isApproved"]
    approved_ids = set(appr_rows.loc[appr_rows["Value"].astype(str) == "True", "Patent_ID"].astype(str))
    duptype_rows = df.loc[df["Field"] == "duplicateType"]
    exact_dup_ids = set(
        duptype_rows.loc[duptype_rows["Value"].map(_strip_label) == "2", "Patent_ID"].astype(str)
    )
    check_ids = approved_ids - exact_dup_ids

    is_optional = df["Field"].astype(str).str.lower().str.contains("|".join(_OPTIONAL_FIELD_HINTS))
    label_rows = df.loc[
        df["Section"].isin(["G1", "M1", "M2", "M3"]) & ~is_optional
        & df["Patent_ID"].astype(str).isin(check_ids)
    ]
    blank_rows = label_rows.loc[label_rows["Value"].isna()]

    incomplete = blank_rows.groupby(blank_rows["Patent_ID"].astype(str))["Field"].apply(
        lambda fields: ", ".join(sorted(set(fields)))
    )

    for pid, fields in incomplete.items():
        _append_flag(df, [pid], f"Incomplete: {fields};")

    print(f"Completeness check: {len(check_ids)} approved non-exact-duplicate patent(s) checked | "
          f"{len(incomplete)} with a blank required field")
    if len(incomplete):
        for pid, fields in incomplete.items():
            print(f"  ⚠ {pid}: {fields}")

    return df


### Orchestration — interactive, per-change confirmation

**Nothing is applied automatically** (unless `HEADLESS=True`, Section 1 — then
every rule is auto-applied to every patent so the notebook can run
unattended). The `RuleReviewPipeline` below walks the rules one at a time,
in order (see `PIPELINE_RULES`):

0. Ghost-block cleanup → **TR. `topType` TP→TR codebook rename (all batches)**
→ A. Combined-Thrust flag → C. `UAVSimilar` propagation (D3-excluded, Bug-2
fix) → D. duplicate inheritance → A1. fixed-arch `acState`→`HoverCruise`
auto-set (Bug-1 fix pt.1) → E. `acState` Ground→Other, convertible-arch only
(Bug-1 fix pt.2) → New. multi-architecture `aircraftName_perArch` suffix
(Part C).

**Removed 2026-08-23.** Step 0b (`empKin`→`empTilts`), Step 0c (Draft→`acSty`),
Rule B and its retroactive check, Rule F and Rule H (`boomXFormat`), Rule G
(`fusKin` "Variable") and **A3** (TB→`fusKin=TiltBody`) are gone from this
notebook. The first six only ever fired on a pre-v15_3 export, and those
batches now go through `02a_legacy.ipynb`, whose wizard round-trip performs
the same migrations using the wizard's own code — keeping a second
implementation here would mean two things that have to agree with no test that
they do. **A3 was deleted rather than moved:** it force-overwrote `fusKin` on
every TB patent, and under the current codebook `LevelCabin` is a legitimate
answer for a TB, so A3 would erase exactly the answers the review queue exists
to collect. Do not reintroduce anything with that behaviour.

For each rule it does a **dry run**, shows every patent the rule would touch
with a readable diff (`Field: old → new`, `+ row added`, `− row removed`,
`flag: ...`), and gives you a checkbox per patent (all ticked by default):

- **Apply selected → next rule** — only the ticked patents get the rule's
  changes; unticked patents keep their current rows untouched.
- **Skip this rule** — nothing applied, move on.

The pipeline works on its own copy — your `df` is only updated when you run
the `df = pipeline.result()` cell after finishing. The completeness check
(report-only, changes nothing) runs inside that same cell.

Rule A is still flag-only here: applying it just marks the patent for the
Section 5 review queue, where you make the actual call with the drawing in
front of you. A4 (assert zero `duplicateType == '4'`) runs earlier,
in Section 3, right after loading — before any of the rules above, since a
nonzero count stops the notebook outright rather than flowing through the
pipeline.


In [ ]:
def _drop_ghost_image_blocks(df: pd.DataFrame) -> pd.DataFrame:
    """Step 0 — drop stale duplicate rows within an image block.

    The wizard can leave a stub behind when a figure is re-keyed (same
    Patent_ID + "Image: <file>" Sub_Dimension, same Field appearing twice
    with different values). Blocks are written in save order, so the LAST
    occurrence is the current one — keep it, drop earlier ones.
    """
    df = df.copy()
    is_img = df["Sub_Dimension"].astype(str).str.startswith("Image: ")
    key_cols = ["Patent_ID", "Sub_Dimension", "Field"]
    dup_mask = is_img & df.duplicated(subset=key_cols, keep="last")
    if dup_mask.any():
        ghost_patents = df.loc[dup_mask, "Patent_ID"].unique().tolist()
        df = df.loc[~dup_mask].reset_index(drop=True)
        _append_flag(df, ghost_patents, "Stale image block removed;")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# TR — architecture code 'TP' renamed 'TR' (Tilt Rotor) to match the codebook.
#
# Runs on EVERY batch, not just the legacy ones. Batch_02 was labelled on a wizard
# that still wrote 'TP' (63 records); Batch_01 has 58 and Batch_05 37. The wizard
# maps TP->TR on LOAD, so anything re-saved through it is already correct — but a
# batch that never gets reloaded (which is every batch, normally) keeps the old
# code, and then topType is not comparable across the corpus. 158 records total.
#
# Value is the "ID — Label" composite, so both halves are rewritten.
TR_VALUE = "TR — Vectored Thrust — Tilt Rotor"


def _rename_tp_to_tr(df: pd.DataFrame) -> pd.DataFrame:
    """Rename architecture code TP -> TR wherever it appears as a topType value."""
    df = df.copy()
    mask = (df["Field"] == "topType") & (df["Value"].map(_strip_label) == "TP")
    if mask.any():
        ids = df.loc[mask, "Patent_ID"].unique().tolist()
        df.loc[mask, "Value"] = TR_VALUE
        _append_flag(df, ids, "topType TP → TR (codebook rename);")
        print(f"TR: {int(mask.sum())} topType row(s) renamed TP → TR "
              f"across {len(ids)} architecture record(s)")
    else:
        print("TR: no 'TP' topType values found — nothing to rename")
    # imgApparentArch mirrors the topType vocabulary, so it carries the old code too.
    m2 = (df["Field"] == "imgApparentArch") & (df["Value"].map(_strip_label) == "TP")
    if m2.any():
        df.loc[m2, "Value"] = TR_VALUE
        print(f"TR: {int(m2.sum())} imgApparentArch row(s) also renamed")
    return df


PIPELINE_RULES = [
    ("0 — Ghost-block cleanup",          _drop_ghost_image_blocks),
    ("TR — topType TP → TR codebook rename (all batches)", _rename_tp_to_tr),
    ("A — Combined Thrust (flag for Section 5)", _flag_combined_thrust),
    ("C — UAVSimilar duplicate-chain propagation (Bug-2 fix: D3 excluded as destination)", _propagate_duplicate_tag),
    ("D — Duplicate inheritance (topType/isApproved)", _inherit_from_duplicate_root),
    ("A1 — Fixed-arch acState auto-set to HoverCruise (Bug-1 fix pt.1)", _force_hovercruise_fixed_arch),
    ("E — acState Ground → Other (Bug-1 fix pt.2: convertible-arch only)", _remap_ground_to_other),
    ("New — multi-architecture aircraftName_perArch suffix (Part C)", _suffix_multiarch_aircraft_names),
]

_DIFF_COLS = ["Section", "Sub_Dimension", "Field", "Value"]


def _patent_rows(df: pd.DataFrame, pid: str) -> pd.DataFrame:
    return df.loc[df["Patent_ID"].astype(str) == pid]


def _flags_of(df: pd.DataFrame, pid: str) -> set:
    if "pre_process_flags" not in df.columns:
        return set()
    vals = _patent_rows(df, pid)["pre_process_flags"].fillna("").astype(str)
    out = set()
    for v in vals:
        out |= {m.strip() for m in v.split(";") if m.strip()}
    return out


def _affected_patents(before: pd.DataFrame, after: pd.DataFrame) -> list:
    """Patent ids whose rows OR flags differ between before and after."""
    def sig(d):
        cols = [c for c in _DIFF_COLS]
        out = {}
        for pid, g in d.groupby(d["Patent_ID"].astype(str)):
            rows = sorted(map(tuple, g[cols].astype(str).values))
            flags = tuple(sorted(_flags_of(d, pid)))
            out[pid] = (tuple(rows), flags)
        return out
    b, a = sig(before), sig(after)
    return sorted(pid for pid in set(b) | set(a) if b.get(pid) != a.get(pid))


# Human-readable names for the field ids labellers actually see in the widget.
# Falls back to the raw field id (still readable, just less pretty) for
# anything not listed here, so a new field never breaks the description.
_FIELD_LABELS = {
    "topType": "Architecture (topType)", "empTilts": "Empennage Tilts",
    "empKin": "Empennage Kinematics (old)", "figKey": "Figure Number",
    "status": "Image Approval Status", "isMain": "Main Figure",
    "acState": "Flight State", "boomXFormat": "Booms X-Formation",
    "boomNotes": "Booms Notes", "duplicateType": "Duplicate Type",
    "isApproved": "Patent Approved", "isDuplicate": "Is Duplicate",
    "duplicateId": "Duplicate Of (Patent ID)", "t1EdgeTags": "Edge-Case Tags",
    "fusKin": "Fuselage Kinematics", "wCount": "Wing Count",
    "latSym": "Laterally Symmetric", "longSym": "Longitudinally Symmetric",
    "qualityFlag": "Figure Quality Flag", "acSty": "Aircraft Rendering Style",
}


def _field_label(field: str) -> str:
    if field in _FIELD_LABELS:
        return _FIELD_LABELS[field]
    if field.endswith("_propKin"):
        card = field[: -len("_propKin")].replace("_", " ").title()
        return f"{card} — Propulsor Kinematics"
    return field


def _describe_change(before: pd.DataFrame, after: pd.DataFrame, pid: str) -> str:
    """Readable per-patent diff: value changes, added/removed rows, new flags.

    Plain-language field names (see _field_label) instead of raw ids, and
    "removed, nothing added" rows for the SAME image block are grouped into
    one summary sentence (Ghost-block cleanup drops a whole stale duplicate
    block at once — showing each of its fields as a separate cryptic "− ..."
    line reads like data loss when it's really one superseded entry going away).
    """
    from collections import Counter
    b = _patent_rows(before, pid)[_DIFF_COLS].astype(str)
    a = _patent_rows(after, pid)[_DIFF_COLS].astype(str)
    bc, ac = Counter(map(tuple, b.values)), Counter(map(tuple, a.values))
    removed = list((bc - ac).elements())
    added = list((ac - bc).elements())

    msgs = []
    rem_by_key = {}
    for sec, sub, field, val in removed:
        rem_by_key.setdefault((sec, sub, field), []).append(val)
    for sec, sub, field, val in added:
        olds = rem_by_key.get((sec, sub, field))
        if olds:
            msgs.append(f"{_field_label(field)} changed: '{olds.pop(0)}' → '{val}'")
            if not olds:
                del rem_by_key[(sec, sub, field)]
        else:
            msgs.append(f"+ Added {_field_label(field)} = '{val}'")

    # Group whatever's left (removed with no matching add) by (Section, Sub_Dimension).
    leftover_by_block = {}
    for (sec, sub, field), olds in rem_by_key.items():
        for val in olds:
            leftover_by_block.setdefault((sec, sub), []).append((field, val))

    for (sec, sub), field_vals in leftover_by_block.items():
        if sub.startswith("Image: ") and len(field_vals) > 1:
            detail = ", ".join(f"{_field_label(f)}='{v}'" for f, v in field_vals)
            # Same figure file can appear twice in the raw export when a
            # labeller re-keys/re-reviews it later in a separate session —
            # the OLD entry (usually incomplete/disapproved) is what's being
            # removed here. Look up whether a CURRENT entry for the same
            # image still exists in `after`, so the labeller can see at a
            # glance that their later, real review survives untouched.
            surviving = after.loc[
                (after["Patent_ID"].astype(str) == pid)
                & (after["Section"] == sec) & (after["Sub_Dimension"] == sub)
            ]
            surv_status = surviving.loc[surviving["Field"] == "status", "Value"]
            surv_figkey = surviving.loc[surviving["Field"] == "figKey", "Value"]
            if not surviving.empty:
                bits = []
                if len(surv_figkey): bits.append(f"Figure Number='{surv_figkey.iloc[0]}'")
                if len(surv_status): bits.append(f"Image Approval Status='{surv_status.iloc[0]}'")
                survives = f" — this image was re-keyed/re-reviewed later; the CURRENT, KEPT entry for it ({', '.join(bits)}) is untouched." if bits else \
                           " — this image was re-keyed/re-reviewed later; a current entry for it is untouched."
            else:
                survives = " (no current entry for this image survives — check it wasn't accidentally lost)"
            msgs.append(f"Removed an OLD, superseded entry for {sub} (it had {detail}){survives}")
        else:
            for field, val in field_vals:
                msgs.append(f"− Removed {_field_label(field)} (was '{val}')")

    for f in sorted(_flags_of(after, pid) - _flags_of(before, pid)):
        msgs.append(f'Flag added: "{f.rstrip(";")}"')
    return "; ".join(msgs) if msgs else "(no visible change)"


def _merge_selected(before: pd.DataFrame, after: pd.DataFrame, selected: set) -> pd.DataFrame:
    """Take `after`'s rows for selected patents, `before`'s for everyone else,
    restoring the original patent order (rules only ever touch a patent's own
    rows, so per-patent row swapping is safe)."""
    for d in (before, after):
        if "pre_process_flags" not in d.columns:
            d["pre_process_flags"] = ""
    b_pid = before["Patent_ID"].astype(str)
    a_pid = after["Patent_ID"].astype(str)
    merged = pd.concat([before[~b_pid.isin(selected)], after[a_pid.isin(selected)]],
                       ignore_index=True)
    order = {pid: i for i, pid in enumerate(dict.fromkeys(b_pid))}
    merged["_o"] = merged["Patent_ID"].astype(str).map(lambda p: order.get(p, len(order)))
    merged = merged.sort_values("_o", kind="stable").drop(columns="_o").reset_index(drop=True)
    return merged



# ── thumbnails for the confirmation rows ─────────────────────────────────────
from io import BytesIO
from PIL import Image as _PILImage

_PIPELINE_MATCHED_DIR = Path(cfg["paths"]["matched"]) / sheet_name
_THUMB_CACHE: dict = {}


def _t2_image_for_patent(df: pd.DataFrame, patent_id: str):
    """Best (Image_Path, label) for THIS architecture's OWN reviewed T2 data.

    Multi-architecture patents keep every T2 (figure) row on the BASE
    patent id only — "_arch2"/"_arch3" ids carry no T2 rows of their own —
    but each figure block IS tagged with which architecture it belongs to
    (Field="arch", values 1..N, matching parse_arch_id()'s arch number).
    Filters to the figures actually tagged for this architecture first.

    Returns (None, None) if this patent has no T2 rows at all (e.g. an
    exact-duplicate patent, which by design was never independently
    reviewed) — callers must NOT silently show a random disk file in that
    case without labelling it as unreviewed (see _resolve_patent_image).
    """
    base_id, arch_num = proc.parse_arch_id(patent_id)
    t2 = df.loc[(df["Patent_ID"].astype(str) == base_id) & (df["Section"] == "T2")
                & df["Image_Path"].notna()]
    if t2.empty:
        return None, None

    arch_tag_rows = df.loc[(df["Patent_ID"].astype(str) == base_id) & (df["Field"] == "arch")]
    arch_by_block = arch_tag_rows.set_index("Sub_Dimension")["Value"]

    def _arch_of(sub):
        v = arch_by_block.get(sub)
        try:
            return int(_strip_label(v)) if v is not None and pd.notna(v) else None
        except (TypeError, ValueError):
            return None

    t2 = t2.copy()
    t2["_arch_tag"] = t2["Sub_Dimension"].map(_arch_of)
    if t2["_arch_tag"].notna().any():
        this_arch = t2[t2["_arch_tag"] == arch_num]
        if not this_arch.empty:
            t2 = this_arch  # else: nothing tagged for this arch — fall back to all figures below

    is_main = t2.loc[(t2["Field"] == "isMain")
                     & t2["Value"].astype(str).str.lower().isin(["true", "1", "yes", "main"]),
                     "Image_Path"]
    if not is_main.empty:
        return Path(str(is_main.iloc[0])), "✅ Approved main figure"
    any_path = t2["Image_Path"].dropna()
    if not any_path.empty:
        return Path(str(any_path.iloc[0])), "Reviewed figure (not marked main)"
    return None, None


def _resolve_patent_image(df: pd.DataFrame, pid: str, matched_dir_root: Path):
    """(path_or_None, provenance_label) for the thumbnail/preview shown to
    the reviewer — never silently passes off an unreviewed raw file as if
    it were an approved figure.

    Order: (1) this patent's own reviewed T2 data; (2) if it's a duplicate
    with no T2 of its own (the normal case for duplicateType "2" — "Images
    AND aircraft the same" — which is never independently reviewed by
    design), follow duplicateId to the chain root and show THAT patent's
    approved figure instead, clearly labelled as inherited; (3) only as a
    last resort, an arbitrary raw file from disk, clearly labelled as
    unreviewed so it can't be mistaken for an approved main image.
    """
    path, label = _t2_image_for_patent(df, pid)
    if path is not None:
        return path, label

    dup_rows = df.loc[(df["Patent_ID"].astype(str) == pid) & (df["Field"] == "duplicateId")
                       & df["Value"].notna() & (df["Value"].astype(str).str.strip() != "")]
    if not dup_rows.empty:
        dup_target = df.loc[(df["Field"] == "duplicateId") & df["Value"].notna()]
        dup_target = dup_target.set_index(dup_target["Patent_ID"].astype(str))["Value"]
        root = _resolve_chain_root(pid, dup_target)
        if root != pid:
            root_path, root_label = _t2_image_for_patent(df, root)
            if root_path is not None:
                return root_path, f"⚠ No figure of its own (duplicate) — showing ORIGINAL {root}'s {root_label.lower()}"

    base = pid.split("_arch")[0]
    try:
        hits = sorted(matched_dir_root.glob(f"{base}_*/*"))
    except OSError:
        hits = []
    if hits:
        return hits[0], "⚠ UNREVIEWED raw file from disk — NOT an approved figure"
    return None, None


def _patent_thumbnail(df: pd.DataFrame, pid: str, max_px: int = 480):
    """(PNG bytes or None, provenance label or None) preview for pid, via
    _resolve_patent_image — labelled so an unreviewed/inherited fallback is
    never mistaken for an approved figure. max_px=480 sets the LONG edge;
    the short edge is whatever the image's real aspect ratio gives it (a
    fixed width previously squashed wide landscape crops into thin strips)."""
    if pid in _THUMB_CACHE:
        return _THUMB_CACHE[pid]
    path, label = _resolve_patent_image(df, pid, _PIPELINE_MATCHED_DIR)
    data = None
    if path is not None and path.exists():
        try:
            im = _PILImage.open(path); im.thumbnail((max_px, max_px))
            buf = BytesIO(); im.convert("RGB").save(buf, "PNG")
            data = buf.getvalue()
        except Exception:
            data = None
    _THUMB_CACHE[pid] = (data, label)
    return data, label


class RuleReviewPipeline:
    """Step through PIPELINE_RULES; every proposed change needs your tick."""

    def __init__(self, df: pd.DataFrame, rules=None, auto_apply: bool = False):
        self.df = df.copy()
        self.rules = list(rules or PIPELINE_RULES)
        self.step = 0
        self.finished = False
        self.auto_apply = auto_apply   # True → apply every rule to every patent, no widget
        self._after = None
        self._boxes = {}

        self.title = widgets.HTML()
        self.list_box = widgets.VBox(
            layout=widgets.Layout(max_height="900px", overflow_y="auto",
                                  border="1px solid #ccc", padding="6px"))
        self.all_btn = widgets.Button(description="Select all")
        self.none_btn = widgets.Button(description="Select none")
        self.apply_btn = widgets.Button(description="Apply selected → next rule",
                                        button_style="success")
        self.skip_btn = widgets.Button(description="Skip this rule (apply nothing)",
                                       button_style="warning")
        self.log = widgets.HTML()
        self.all_btn.on_click(lambda b: self._set_all(True))
        self.none_btn.on_click(lambda b: self._set_all(False))
        self.apply_btn.on_click(self._on_apply)
        self.skip_btn.on_click(self._on_skip)
        self.root = widgets.VBox([self.title,
                                  widgets.HBox([self.all_btn, self.none_btn]),
                                  self.list_box,
                                  widgets.HBox([self.apply_btn, self.skip_btn]),
                                  self.log])
        self._render_step()

    def _set_all(self, value):
        for box in self._boxes.values():
            box.value = value

    def _render_step(self):
        while self.step < len(self.rules):
            name, fn = self.rules[self.step]
            self._after = fn(self.df)
            affected = _affected_patents(self.df, self._after)
            if not affected:
                self.log.value += f"<div>• <b>{name}</b>: nothing to change — auto-skipped.</div>"
                self.step += 1
                continue
            if self.auto_apply:
                self.df = _merge_selected(self.df, self._after, set(affected))
                self.log.value += (f"<div>• <b>{name}</b>: AUTO-applied to "
                                   f"{len(affected)} patent(s) (headless).</div>")
                print(f"  [headless] {name}: auto-applied to {len(affected)} patent(s)")
                self.step += 1
                continue
            self.title.value = (f"<h4>Rule {name}</h4><b>{len(affected)}</b> patent(s) "
                                f"would change — untick anything you do NOT want applied "
                                f"(step {self.step + 1}/{len(self.rules)}):")
            self._boxes = {}
            # One INDEPENDENT HBox per patent (not a shared GridBox) — each
            # row lays itself out with no cross-row height-sharing, which is
            # what broke: GridBox's row height came from its tallest cell,
            # but different figures have very different aspect ratios, so
            # rows ended up misaligned between the image and text columns.
            # widgets.Image (not raw HTML <img>, which VSCode collapsed to
            # scrollbar bars) + explicit flex-basis per column (the fix that
            # stopped the image container from stretching past its image).
            items = []
            for pid in affected:
                box = widgets.Checkbox(value=True, indent=False,
                                       layout=widgets.Layout(width="28px", flex="0 0 28px"))
                self._boxes[pid] = box
                desc = _describe_change(self.df, self._after, pid)
                thumb, thumb_label = _patent_thumbnail(self.df, pid)
                if thumb:
                    img = widgets.Image(value=thumb, format="png",
                                        layout=widgets.Layout(max_width="420px",
                                                              max_height="420px"))
                    warn = bool(thumb_label) and thumb_label.startswith("⚠")
                    cap = widgets.HTML(
                        f"<span style='font-size:11px;color:{'#b45309' if warn else '#555'}'>"
                        f"{thumb_label or ''}</span>")
                    img_col = widgets.VBox([img, cap],
                                           layout=widgets.Layout(flex="0 0 440px",
                                                                 align_items="flex-start"))
                else:
                    img_col = widgets.HTML("<i style='color:#999'>no image</i>",
                                           layout=widgets.Layout(flex="0 0 440px"))
                text_w = widgets.HTML(f"<b>{pid}</b> — {desc}",
                                      layout=widgets.Layout(margin="0 0 0 12px",
                                                            min_width="300px", flex="1 1 auto"))
                items.append(widgets.HBox(
                    [box, img_col, text_w],
                    layout=widgets.Layout(align_items="flex-start",
                                          border_bottom="1px solid #eee",
                                          padding="10px 0", width="100%")))
            self.list_box.children = items
            return
        self._finish()

    def _on_apply(self, _btn):
        name, _ = self.rules[self.step]
        selected = {pid for pid, box in self._boxes.items() if box.value}
        self.df = _merge_selected(self.df, self._after, selected)
        n_rej = len(self._boxes) - len(selected)
        self.log.value += (f"<div>• <b>{name}</b>: applied to {len(selected)} patent(s)"
                           + (f", rejected {n_rej}" if n_rej else "") + ".</div>")
        self.step += 1
        self._render_step()

    def _on_skip(self, _btn):
        name, _ = self.rules[self.step]
        self.log.value += f"<div>• <b>{name}</b>: skipped (nothing applied).</div>"
        self.step += 1
        self._render_step()

    def _finish(self):
        self.finished = True
        self.title.value = "<h4>✅ All rules reviewed.</h4>Run the next cell: <code>df = pipeline.result()</code>"
        self.list_box.children = []
        self.apply_btn.disabled = self.skip_btn.disabled = True
        self.all_btn.disabled = self.none_btn.disabled = True

    def result(self) -> pd.DataFrame:
        assert self.finished, "Finish reviewing all rules in the widget first."
        return self.df.copy()

    def display(self):
        display(self.root)


pipeline = RuleReviewPipeline(df, auto_apply=HEADLESS)
if HEADLESS:
    print("HEADLESS: all rules auto-applied above — no widget to click.")
else:
    pipeline.display()


In [ ]:
# Commits the rule pipeline's changes to df and runs the report-only completeness
# check (flags gaps, changes no labels). Runs BEFORE Section 5's review queue.
df = pipeline.result()
df = _check_label_completeness(df)

# Per-flag patent counts from the rule pipeline. Section 5 walks a separate,
# codebook-driven queue (src/review_worklists.py); these flags are the rules' own
# report and are carried into the export's pre_process_flags column.
_per_flag, _flagged_patents = {}, 0
for pid, g in df.groupby(df["Patent_ID"].astype(str)):
    flags = {m.strip() for v in g["pre_process_flags"].fillna("").astype(str)
             for m in v.split(";") if m.strip()}
    if flags:
        _flagged_patents += 1
    for f in flags:
        _per_flag[f] = _per_flag.get(f, 0) + 1
print(f"{_flagged_patents} patent(s) carry at least one flag:")
for f, n in sorted(_per_flag.items(), key=lambda kv: -kv[1]):
    print(f"  {n:4d}  {f}")


# ── Addendum (2026-07-21): S3 + D5 — report-only, run on the same confirmed ──
# post-pipeline df as the completeness check above.
#
# The A5 boomXFormat-conflict worklist and the Draft-migration worklists that used
# to sit here are GONE. Both only ever fired on a pre-v15_3 export, and those
# batches now go through 02a_legacy, whose wizard round-trip performs the same
# migrations from the wizard's own code. Keeping a second implementation here
# would mean two things that have to agree and no test that they do.
emptilts_rigid_arch_worklist, emptilts_missing_justification = _check_emptilts_on_rigid_arch(df)
confidence_constancy_report = _check_confidence_constancy(df)


# Surface the actual worklist rows (not just counts) so there is something to
# act on — a count alone is not a worklist a human can resolve against.
if len(emptilts_rigid_arch_worklist):
    print("emptilts_rigid_arch_worklist:")
    display(emptilts_rigid_arch_worklist)
if len(emptilts_missing_justification):
    print("emptilts_missing_justification (escalated — real data-quality issue):")
    display(emptilts_missing_justification)


# ── Part A/B/C change-log (2026-07-22) — bug fixes + still-missing tasks + ──
# new codebook-pass task. Report-only: pulled from flags/rows already written
# by the pipeline above onto the confirmed post-pipeline df, formatted so the
# counts can be pasted straight into the thesis change-log table.
def _flag_patent_count(flag_text: str) -> int:
    return _per_flag.get(flag_text, 0)


# Bug-2 D3-exclusion count recomputed fresh against the final df (Rule C's
# own print already showed this during the pipeline step; repeated here so
# it's captured in the same end-of-run block as everything else).
_dup_edges = df.loc[(df["Field"] == "duplicateId") & df["Value"].notna()
                    & (df["Value"].astype(str).str.strip() != "")]
_graph = nx.Graph()
_graph.add_nodes_from(df["Patent_ID"].astype(str).unique())
for _, _row in _dup_edges.iterrows():
    _graph.add_edge(str(_row["Patent_ID"]), str(_row["Value"]).strip())
_edge_tag_rows = df["Field"] == "t1EdgeTags"
_tagged_ids = set(df.loc[_edge_tag_rows & df["Value"].astype(str).str.contains("UAVSimilar", na=False),
                         "Patent_ID"].astype(str))
_dt_rows = df.loc[df["Field"] == "duplicateType"]
_dt_by_patent = _dt_rows.set_index(_dt_rows["Patent_ID"].astype(str))["Value"].map(_strip_label)
_d3_ids = set(_dt_by_patent[_dt_by_patent == "3"].index)
_d3_excluded = set()
for _component in nx.connected_components(_graph):
    if len(_component) > 1 and (_component & _tagged_ids):
        _d3_excluded |= ((_component - _tagged_ids) & _d3_ids)

print("\n" + "=" * 70)
print(f"PART A/B/C CHANGE-LOG ({datetime.now().date().isoformat()})")
print("=" * 70)
print(f"Bug-1 (A1, acState fixed-arch auto-set): {_flag_patent_count('acState forced to HoverCruise (fixed arch)')} patent(s)")
print(f"Bug-1 (Rule E narrowed, convertible-arch Ground→Other):        {_flag_patent_count('acState Ground → Other')} patent(s)")
print(f"Bug-2 (Rule C, UAVSimilar propagated):                         {_flag_patent_count('UAV Tag Propagated')} patent(s)")
print(f"Bug-2 (Rule C, D3 excluded from propagation destination):      {len(_d3_excluded)} patent(s) {sorted(_d3_excluded) if _d3_excluded else ''}")
print(f"A3 (explicit prior VarInc — flagged for a second look):        {_flag_patent_count('A3: fusKin was explicitly VarInc — double-check TB override')} patent(s)")
print(f"A4 (assert zero duplicateType == '4'):                         PASSED (ran at load time, Section 3 — see cell output above)")
print(f"New/Part C (aircraftName_perArch multi-arch suffix):           {_flag_patent_count('aircraftName_perArch suffix added (multi-architecture)')} patent(s)")
print("=" * 70)


## Section 5 — Codebook review queue

Everything a **rule** can settle has already been applied by the time this runs. What is
left is the set of decisions that need the drawing, built by `src/review_worklists.py`:

| worklist | what it asks |
|---|---|
| `A-9` | tightened L5 — a Tilt Wing must have EVERY propulsor tilt with the wing |
| `A-13` | empennage vs fuselage — an empennage bears stabilising surfaces |
| `bodyMotion` | what the BODY does between hover and cruise (TB, or a legacy `VarInc`/`Variable`) |
| `acState` | figures still holding the retired `Unclear` / `NonApplicable` |
| `rmech` | `Retractable` was split into blade-folding vs retracting |
| `propKin` | `Cyclic` / `Vectored` retired |
| `fusShape` | `PodBoom` retired |
| `bgSty` | `Grid/Pattern` retired |
| `t1Reason` | the retired `Unreadable/Insufficient image quality` disapproval reason, plus `Other` disapprovals whose note says the figures were inadequate |

`t1Reason` is the only worklist that looks at **disapproved** patents — every other
question skips them, because they never reach 02b. This one exists precisely because
the disapproval *reason* is what is being corrected. Those records carry almost no T2
rows, so their figures are resolved straight off `00b2_figure_crops/` instead.

**Nothing here is auto-answered.** The queue proposes, you decide, and Section 5c writes
only what you decided. That is what keeps these human annotations rather than machine
labels — a value a script invented and nobody confirmed would have to be disclosed as
such and dropped from the intra-annotator kappa analysis.

**Keys:** `1`–`9` decide and advance · `s` skip · `n`/`p` next/prev · `u` undo ·
`e` expand the focused figure · `[` `]` cycle which figure is focused.

Every keystroke is written to `review_decisions_<batch>.csv` immediately, so a kernel
crash costs nothing and re-running this cell resumes exactly where you stopped. Decided
records do not come back.

In [ ]:
from pathlib import Path
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from ipyevents import Event

from src.review_worklists import build_worklist, QUESTIONS

DECISIONS_DIR = Path(cfg["paths"]["base"]) / "data" / "03b_CONFORMED_legacy" / "proposals"
DECISIONS_DIR.mkdir(parents=True, exist_ok=True)

# Absolute pixels, never percentages. The old panel here was
# Output(layout=Layout(width="45%")) with no height, so a figure drew at 45% of a narrow
# column and the row collapsed. These are hover-vs-cruise comparisons — they have to be
# side by side and they have to be big.
FIG_W, GRID_W, GRID_H = 420, 900, 700


class WorklistReview:
    """One decision at a time, every figure of the patent on screen at once."""

    def __init__(self, items, batch, decisions_csv=None):
        self.items = items
        self.batch = batch
        self.csv = Path(decisions_csv or DECISIONS_DIR / f"review_decisions_{batch}.csv")
        self.decisions = {}
        self.order = []          # uids, most recent last — drives undo
        if self.csv.exists():
            prev = pd.read_csv(self.csv)
            for _, r in prev.iterrows():
                self.decisions[r["uid"]] = r.to_dict()
                self.order.append(r["uid"])
            print(f"resumed {len(self.decisions)} decision(s) from {self.csv.name}")
        self.cur = 0
        self.focus = 0
        self._build()

    # ── queue ────────────────────────────────────────────────────────────────
    def todo(self):
        return [i for i in self.items if i["uid"] not in self.decisions]

    def item(self):
        q = self.todo()
        if not q:
            return None
        self.cur = max(0, min(self.cur, len(q) - 1))
        return q[self.cur]

    # ── layout ───────────────────────────────────────────────────────────────
    def _build(self):
        self.head = widgets.HTML()
        self.figbox = widgets.Box(layout=widgets.Layout(
            width=f"{GRID_W}px", min_height=f"{GRID_H}px", flex_flow="row wrap",
            align_content="flex-start", overflow_y="auto", border="1px solid #d0d0d0"))
        self.zoom = widgets.Box(layout=widgets.Layout(width=f"{GRID_W}px"))
        self.qbox = widgets.HTML()
        self.btns = widgets.VBox()
        self.meta = widgets.HTML(layout=widgets.Layout(width="440px"))
        right = widgets.VBox([self.qbox, self.btns, self.meta],
                             layout=widgets.Layout(width="460px", margin="0 0 0 14px"))
        self.root = widgets.VBox(
            [self.head, widgets.HBox([widgets.VBox([self.figbox, self.zoom]), right],
                                     layout=widgets.Layout(align_items="flex-start"))])
        # ipyevents needs a focusable node; the container gets a tabindex so keydown
        # lands here instead of scrolling the notebook.
        self.root.add_class("worklist-root")
        self.ev = Event(source=self.root, watched_events=["keydown"], prevent_default_action=True)
        self.ev.on_dom_event(self._key)
        self.render()

    def _figures(self, it):
        kids = []
        for n, f in enumerate(it["figures"]):
            p = Path(f["path"])
            box = [widgets.HTML(f"<div style='font:600 13px sans-serif;text-align:center;"
                                f"padding:2px'>{f['fig']}{' ◀' if n == self.focus else ''}</div>")]
            if p.exists():
                box.insert(0, widgets.Image(value=p.read_bytes(), format=p.suffix.lstrip(".") or "png",
                                            layout=widgets.Layout(width=f"{FIG_W}px", height="auto")))
            else:
                box.insert(0, widgets.HTML(
                    f"<div style='width:{FIG_W}px;height:180px;display:flex;align-items:center;"
                    f"justify-content:center;background:#eee;color:#888'>image not on disk</div>"))
            kids.append(widgets.VBox(box, layout=widgets.Layout(
                margin="6px", border=("2px solid #1f77b4" if n == self.focus else "1px solid #ddd"))))
        return kids

    def render(self):
        it = self.item()
        total, left = len(self.items), len(self.todo())
        if it is None:
            self.head.value = (f"<h3 style='font-family:sans-serif'>Queue empty — "
                               f"{total} decision(s) recorded.</h3>"
                               f"<div style='color:#666'>Run Section 5c to apply them.</div>")
            self.figbox.children, self.btns.children = (), ()
            self.qbox.value = self.meta.value = ""
            return
        qd = QUESTIONS[it["q"]]
        self.head.value = (
            f"<div style='font:15px sans-serif;padding:6px 0'>"
            f"<b style='font-size:20px'>{total-left} / {total}</b>"
            f" &nbsp;·&nbsp; <span style='color:#b8860b'>{it['q']} — {qd['label']}</span>"
            f" &nbsp;·&nbsp; <code>{it['patent']}</code> ({it['batch']})"
            f"{' · ' + it['slot'] if it['slot'] else ''}</div>")
        self.figbox.children = self._figures(it)
        self.zoom.children = ()
        self.qbox.value = (
            f"<div style='font:600 17px sans-serif;margin-bottom:4px'>{it['title'][:110]}</div>"
            f"<div style='background:#fff8e1;border-left:3px solid #f0b429;padding:8px 10px;"
            f"font:14px sans-serif;white-space:pre-wrap'>{it['why']}</div>")

        # Only the field under question. >=15px. The value the record holds is marked.
        rows = []
        for n, (oid, prose) in enumerate(qd["options"], 1):
            is_cur = str(it["old"]) == str(oid)
            b = widgets.Button(description=f"{n}. {prose}"[:78],
                               tooltip=f"press {n}",
                               layout=widgets.Layout(width="440px", height="auto",
                                                     margin="3px 0"),
                               style=dict(font_size="15px", button_color="#e8f0fe" if is_cur else None))
            b.on_click(lambda _b, v=oid: self.decide(v))
            rows.append(b)
            if is_cur:
                rows.append(widgets.HTML("<div style='font:12px sans-serif;color:#888;"
                                         "margin:-2px 0 4px 6px'>▲ current value</div>"))
        self.btns.children = rows
        self.meta.value = (
            f"<div style='font:13px sans-serif;color:#444'>"
            f"<h4 style='margin:12px 0 2px'>Abstract</h4>"
            f"<div style='max-height:150px;overflow:auto;white-space:pre-wrap'>{it['abstract'] or '—'}</div>"
            f"<h4 style='margin:12px 0 2px'>Brief description of drawings</h4>"
            f"<div style='max-height:170px;overflow:auto;white-space:pre-wrap'>{it['drawings'] or '—'}</div>"
            f"<div style='margin-top:12px;color:#888'>1–{len(qd['options'])} decide · s skip · "
            f"n/p move · u undo · [ ] focus figure · e expand</div></div>")

    # ── actions ──────────────────────────────────────────────────────────────
    def decide(self, val):
        it = self.item()
        if it is None:
            return
        self.decisions[it["uid"]] = dict(
            uid=it["uid"], batch=it["batch"], patent=it["patent"], question=it["q"],
            slot=it["slot"], sub_dim=it["sub_dim"], field=QUESTIONS[it["q"]]["field"] or "",
            old_value=it["old"], decision=val, decided_at=pd.Timestamp.utcnow().isoformat())
        self.order.append(it["uid"])
        self.save()                      # before anything else — crash-safe
        self.focus = 0
        self.render()

    def undo(self):
        while self.order:
            uid = self.order.pop()
            if uid in self.decisions:
                del self.decisions[uid]
                self.save()
                self.cur = 0
                self.render()
                return

    def save(self):
        pd.DataFrame(list(self.decisions.values())).to_csv(self.csv, index=False)

    def _key(self, e):
        k = str(e.get("key", "")).lower()
        it = self.item()
        if k.isdigit() and k != "0" and it is not None:
            opts = QUESTIONS[it["q"]]["options"]
            if int(k) <= len(opts):
                self.decide(opts[int(k) - 1][0])
        elif k == "s":
            self.decide("__skip__")
        elif k == "n":
            self.cur += 1; self.focus = 0; self.render()
        elif k == "p":
            self.cur -= 1; self.focus = 0; self.render()
        elif k == "u":
            self.undo()
        elif k in ("[", "]") and it is not None and it["figures"]:
            self.focus = (self.focus + (1 if k == "]" else -1)) % len(it["figures"])
            self.render()
        elif k == "e" and it is not None and it["figures"]:
            f = it["figures"][self.focus]
            p = Path(f["path"])
            self.zoom.children = ([widgets.Image(value=p.read_bytes(),
                                                 format=p.suffix.lstrip(".") or "png",
                                                 layout=widgets.Layout(width=f"{GRID_W}px"))]
                                  if p.exists() else ())

    def show(self):
        display(self.root)


worklist = build_worklist(df, sheet_name)
print(f"{sheet_name}: {len(worklist)} decision(s) need a figure")
print(pd.Series([i["q"] for i in worklist]).value_counts().to_string() if worklist else "  (none)")

review = WorklistReview(worklist, sheet_name)
review.show()

### Section 5c — Apply the decisions

Reads `review_decisions_<batch>.csv` and writes **only** what you decided into the
corrected working copy, appending one row per change to `CORRECTIONS_LOG.csv`.

Three classes of decision, handled differently on purpose:

| | applied here | why |
|---|---|---|
| `bodyMotion` `acState` `rmech` `propKin` `fusShape` `bgSty` | **yes** — direct value write | one field, one value, no cross-field consequence |
| `A-13 → fuselage`, fuselage card empty | **yes** — the `emp_*` rows are re-prefixed `fuselage_*` | a clean move, nothing to merge |
| `A-13 → fuselage`, fuselage card already occupied | **no** — listed for the wizard | two propulsor sets would have to be merged and their types reconciled; a script guessing that is how you get silent corruption |
| `A-9 → CVT` | **no** — listed for the wizard | changing `topType` releases `propKinLock`, so `propKin` becomes a real question on every station. A hand-edited `topType` leaves the record carrying values a lock wrote for an architecture it no longer has |

Answers that keep the current value (`A-9 → TW`, `A-13 → empennage`) are recorded as
confirmations and change nothing — which is most of them.

In [ ]:
import re
import shutil
from datetime import datetime, timezone

DEC_CSV = DECISIONS_DIR / f"review_decisions_{sheet_name}.csv"
CORRECTED_DIR = Path(cfg["paths"]["corrected_wizard_exports"])
target_xlsx = CORRECTED_DIR / f"reviewed_patents_{sheet_name}.xlsx"

if not DEC_CSV.exists():
    print(f"no decisions yet at {DEC_CSV.name} — review in Section 5 first")
else:
    dec = pd.read_csv(DEC_CSV)
    dec = dec[dec.decision != "__skip__"]
    tgt = pd.read_excel(target_xlsx, sheet_name="Review")

    applied, wizard, confirmed = [], [], []
    # Concrete ops mirrored onto the in-memory `df` at the tail of this cell.
    # Without it 5c would write the xlsx only, and Section 5b/6 would keep
    # working off the pre-decision `df` — the export would silently contain
    # none of the decisions just made.
    replay = []
    DIRECT = {"bodyMotion", "acState", "rmech", "propKin", "fusShape", "bgSty", "t1Reason"}
    # t1Reason writes t1DisapproveReason, one row per patent, so it takes the plain
    # (Patent_ID, Field) branch below with no slot or sub-dimension.

    def _val_with_label(field_id, question):
        """Store the id alone; the wizard re-labels on load and 02a compares on the id."""
        return field_id

    for _, d in dec.iterrows():
        pid, q, val = d["patent"], d["question"], d["decision"]

        if q == "A-9":
            (confirmed if val == "TW" else wizard).append(
                dict(patent=pid, question=q, decision=val,
                     note="topType change releases propKinLock — re-answer propKin in the wizard"))
            continue

        if q == "A-13":
            if val == "empennage":
                confirmed.append(dict(patent=pid, question=q, decision=val, note="mount confirmed"))
                continue
            fc = tgt[(tgt.Patent_ID == pid) & (tgt.Field == "fuselage_count")]
            n = pd.to_numeric(fc.Value, errors="coerce").fillna(0).sum() if len(fc) else 0
            if n > 0:
                wizard.append(dict(patent=pid, question=q, decision=val,
                                   note=f"fuselage card already carries {n:.0f} propulsor(s) — needs a merge"))
                continue
            # A propulsor count of 0 does NOT mean the fuselage card is absent — the
            # skeleton rows (fuselage_count / _ntypes / _sym) are usually still there.
            # Renaming emp_* on top of them produces DUPLICATE Fields on one patent,
            # which pandas will happily write and 02b will read as whichever comes
            # first. So: overwrite where the target row exists, rename where it does
            # not, and drop the emp_ row in the overwrite case.
            own = tgt.index[tgt.Patent_ID == pid]
            existing = {str(tgt.at[j, "Field"]): j for j in own
                        if str(tgt.at[j, "Field"]).startswith("fuselage_")}
            mask = (tgt.Patent_ID == pid) & tgt.Field.astype(str).str.startswith("emp_")
            drop_idx = []
            for i in tgt.index[mask]:
                old_f = str(tgt.at[i, "Field"])
                new_f = "fuselage_" + old_f[len("emp_"):]
                if new_f in existing:
                    j = existing[new_f]
                    prev = str(tgt.at[j, "Value"])
                    tgt.at[j, "Value"] = tgt.at[i, "Value"]
                    drop_idx.append(i)
                    applied.append(dict(patent=pid, field=new_f, old=prev,
                                        new=str(tgt.at[i, "Value"]), question=q,
                                        note=f"mount moved: {old_f} merged into existing {new_f}"))
                    replay.append(("set", pid, new_f, None, tgt.at[i, "Value"]))
                    replay.append(("drop", pid, old_f))
                else:
                    tgt.at[i, "Field"] = new_f
                    sub = str(tgt.at[i, "Sub_Dimension"])
                    # Targeted: only the card token, so "Empennage"/"emp" elsewhere in the
                    # label is left alone.
                    tgt.at[i, "Sub_Dimension"] = re.sub(r"\bemp\b", "fuselage", sub)
                    applied.append(dict(patent=pid, field=old_f, old=str(tgt.at[i, "Value"]),
                                        new=str(tgt.at[i, "Value"]), question=q,
                                        note=f"mount moved: {old_f} -> {new_f}"))
                    replay.append(("rename", pid, old_f, new_f))
            if drop_idx:
                tgt = tgt.drop(index=drop_idx)
            continue

        if q in DIRECT:
            fld = str(d["field"]) if pd.notna(d["field"]) and d["field"] else None
            slot = str(d["slot"]) if pd.notna(d["slot"]) and d["slot"] else None
            # rmech/propKin/bgSty carry the exact Field (or Sub_Dimension) in `slot`,
            # because one patent can hold several of them on different cards/figures.
            if q in ("rmech", "propKin"):
                mask = (tgt.Patent_ID == pid) & (tgt.Field == slot)
            elif q in ("bgSty", "acState"):
                mask = (tgt.Patent_ID == pid) & (tgt.Field == (fld or q)) & \
                       (tgt.Sub_Dimension.astype(str) == str(d["sub_dim"]))
            else:
                mask = (tgt.Patent_ID == pid) & (tgt.Field == fld)
            idxs = list(tgt.index[mask])
            if not idxs:
                wizard.append(dict(patent=pid, question=q, decision=val,
                                   note="target row not found — check by hand"))
                continue
            for i in idxs:
                old = str(tgt.at[i, "Value"])
                tgt.at[i, "Value"] = val
                applied.append(dict(patent=pid, field=tgt.at[i, "Field"], old=old, new=val,
                                    question=q, note="reviewer decision"))
            # Same targeting the mask above used, recorded once per decision.
            replay.append(("set", pid,
                           slot if q in ("rmech", "propKin") else (fld or q),
                           str(d["sub_dim"]) if q in ("bgSty", "acState") else None,
                           val))

    print(f"applied to the xlsx : {len(applied)} row(s)")
    print(f"confirmed unchanged : {len(confirmed)}")
    print(f"needs the wizard    : {len(wizard)}")
    for w in wizard[:15]:
        print(f"   {w['patent']:26s} {w['question']:11s} -> {w['decision']:10s} {w['note']}")

    if applied:
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        shutil.copy2(target_xlsx, target_xlsx.with_name(f"{target_xlsx.stem}.PRE_REVIEW_{stamp}.xlsx"))
        tgt.to_excel(target_xlsx, sheet_name="Review", index=False)
        now = datetime.now(timezone.utc).isoformat(timespec="seconds")
        log = pd.DataFrame([dict(applied_utc=now, batch=sheet_name, patent_id=a["patent"],
                                 section="", field=a["field"], old_value=a["old"],
                                 new_value=a["new"], rule=f"review worklist {a['question']}",
                                 evidence=a["note"], confirmed_by="annotator (Section 5)")
                            for a in applied])
        lp = CORRECTED_DIR / "CORRECTIONS_LOG.csv"
        log.to_csv(lp, mode="a", header=not lp.exists() or lp.stat().st_size == 0, index=False)
        print(f"\nwrote {target_xlsx.name}  (+{len(log)} rows in {lp.name})")

    if wizard:
        wp = DECISIONS_DIR / f"needs_wizard_{sheet_name}.csv"
        pd.DataFrame(wizard).to_csv(wp, index=False)
        print(f"wizard worklist -> {wp.name}")

    # ── Mirror the same changes onto the in-memory `df` ──────────────────────
    # `df` here is post-Section-4 (rules applied, pre_process_flags column), so it
    # cannot just be reloaded from the xlsx without losing that work — and a
    # reload would also discard any per-patent tick the reviewer unticked in the
    # rule widget. Replaying the ops keeps both. After this, Section 5b and
    # Section 6 see the decisions without re-running anything.
    _n_sync = 0
    for _op in replay:
        if _op[0] == "set":
            _, _pid, _fld, _sub, _val = _op
            _m = (df["Patent_ID"].astype(str) == str(_pid)) & (df["Field"] == _fld)
            if _sub is not None:
                _m &= df["Sub_Dimension"].astype(str) == _sub
            _n_sync += int(_m.sum())
            df.loc[_m, "Value"] = _val
        elif _op[0] == "rename":
            _, _pid, _old_f, _new_f = _op
            _m = (df["Patent_ID"].astype(str) == str(_pid)) & (df["Field"] == _old_f)
            _n_sync += int(_m.sum())
            df.loc[_m, "Sub_Dimension"] = (df.loc[_m, "Sub_Dimension"].astype(str)
                                           .map(lambda s: re.sub(r"\bemp\b", "fuselage", s)))
            df.loc[_m, "Field"] = _new_f
        elif _op[0] == "drop":
            _, _pid, _fld = _op
            _m = (df["Patent_ID"].astype(str) == str(_pid)) & (df["Field"] == _fld)
            _n_sync += int(_m.sum())
            df = df.loc[~_m].reset_index(drop=True)
    print(f"in-memory df synced : {_n_sync} row(s) across {len(replay)} decision(s) "
          f"— Section 5b and Section 6 now see them (no re-run needed)")

## Section 5b — Missing Main-Figure Review

Some patents reach this stage with **zero reviewed T2 images** — every
`status`/`isMain` row is blank because the reviewer never opened the image
step for them in the wizard (found via a completeness check against the
approved-figure count, not a hardcoded list — currently `US2018354616A1` in
Batch_01; `KR102760903B1`, `US2022089276A1`, `US2022348339A1`,
`WO2023272353A1` in Batch_05). Without at least one `status="approved"` +
`isMain=True` image, these patents can't feed the embeddings pipeline.

This section queues exactly those patents (any patent with T2 rows but no
`isMain=True` among them) and lets you, per image: mark **Approved** and
pick the one **Main** figure. **Save & Next** writes the decisions into
`df`'s `status`/`isMain` rows in place (the same in-memory `df` Section 4
operates on) and appends a `pre_process_flags` note — nothing touches disk
until Section 6's export.

In [18]:
def find_missing_main_figure_patents(df: pd.DataFrame) -> list:
    """Patents with T2 image rows but no isMain=True among them.

    Distinct from duplicates (which correctly carry zero T2 rows at all —
    see Rule D above) — this only catches patents whose image-review step
    was itself skipped or left incomplete.
    """
    t2 = df.loc[df["Section"] == "T2"]
    has_t2 = set(t2["Patent_ID"].unique())
    is_main_true = t2.loc[
        (t2["Field"] == "isMain") & (t2["Value"].astype(str).str.lower() == "true"),
        "Patent_ID",
    ]
    return sorted(has_t2 - set(is_main_true.unique()))


class MainFigureSession:
    """One patent at a time: review every T2 image, mark it Approved/not,
    and pick exactly one Main figure. Writes directly into `self.df`.
    """

    def __init__(self, df: pd.DataFrame, patent_ids: list, matched_dir_root: Path):
        self.df = df  # mutated in place by Save & Next
        self.patent_ids = patent_ids
        self.matched_dir_root = matched_dir_root
        self.pos = 0

        self.status_label = widgets.Label()
        self.images_panel = widgets.VBox()
        self.save_btn = widgets.Button(description="Save & Next", button_style="success")
        self.skip_btn = widgets.Button(description="Skip (no change)")
        self.prev_btn = widgets.Button(description="◀ Prev")

        self.save_btn.on_click(self._on_save)
        self.skip_btn.on_click(self._on_skip)
        self.prev_btn.on_click(self._on_prev)

        self.root = widgets.VBox([
            self.status_label,
            self.images_panel,
            widgets.HBox([self.prev_btn, self.skip_btn, self.save_btn]),
        ])
        self._render()

    def _current_patent_id(self):
        return None if not self.patent_ids else self.patent_ids[self.pos]

    def _render(self):
        n = len(self.patent_ids)
        if n == 0:
            self.status_label.value = "No patents need a main-figure review."
            self.images_panel.children = []
            return

        pid = self._current_patent_id()
        self.status_label.value = f"Reviewing {self.pos + 1} of {n} — {pid}"

        t2 = self.df.loc[(self.df["Patent_ID"] == pid) & (self.df["Section"] == "T2")]
        sub_dims = sorted(t2["Sub_Dimension"].dropna().unique())

        main_group = widgets.RadioButtons(options=["(none)"] + sub_dims, value="(none)",
                                           description="Main:", layout=widgets.Layout(width="100%"))
        approve_boxes = {}
        rows = []
        for sub_dim in sub_dims:
            img_rows = t2.loc[t2["Sub_Dimension"] == sub_dim]
            image_path_str = (img_rows["Image_Path"].dropna().iloc[0]
                               if img_rows["Image_Path"].notna().any() else None)
            out = widgets.Output(layout=widgets.Layout(width="220px", border="1px solid #ccc"))
            with out:
                if image_path_str and Path(image_path_str).exists():
                    p = Path(image_path_str)
                    display(widgets.Image(value=p.read_bytes(), format=p.suffix.lstrip(".") or "png",
                                           layout=widgets.Layout(max_width="200px")))
                else:
                    print(f"⚠ no image at {image_path_str}")
            approve_box = widgets.Checkbox(value=False, description="Approved", indent=False)
            approve_boxes[sub_dim] = approve_box
            rows.append(widgets.VBox([out, widgets.Label(sub_dim), approve_box],
                                      layout=widgets.Layout(margin="4px")))

        self._approve_boxes = approve_boxes
        self._main_group = main_group

        grid = widgets.HBox(rows, layout=widgets.Layout(flex_flow="row wrap"))
        self.images_panel.children = [grid, main_group]

    def _write_decisions(self):
        pid = self._current_patent_id()
        if pid is None:
            return
        main_choice = self._main_group.value
        for sub_dim, approve_box in self._approve_boxes.items():
            status_mask = (
                (self.df["Patent_ID"] == pid)
                & (self.df["Sub_Dimension"] == sub_dim)
                & (self.df["Field"] == "status")
            )
            self.df.loc[status_mask, "Value"] = "approved" if approve_box.value else "disapproved"

            is_main_mask = (
                (self.df["Patent_ID"] == pid)
                & (self.df["Sub_Dimension"] == sub_dim)
                & (self.df["Field"] == "isMain")
            )
            self.df.loc[is_main_mask, "Value"] = (sub_dim == main_choice)

        _append_flag(self.df, [pid], "Main figure reviewed in 02a;")

    def _on_save(self, _btn):
        self._write_decisions()
        self._advance()

    def _on_skip(self, _btn):
        self._advance()

    def _on_prev(self, _btn):
        if self.pos > 0:
            self.pos -= 1
            self._render()

    def _advance(self):
        if self.pos < len(self.patent_ids) - 1:
            self.pos += 1
            self._render()
        else:
            self.status_label.value = "Done — no more patents in the queue."
            self.images_panel.children = []

    def display(self):
        display(self.root)


def build_main_figure_review(df: pd.DataFrame, matched_dir_root: Path = Path(cfg["paths"]["matched"])) -> MainFigureSession:
    patent_ids = find_missing_main_figure_patents(df)
    print(f"{len(patent_ids)} patent(s) need a main-figure review: {patent_ids}")
    session = MainFigureSession(df, patent_ids, matched_dir_root)
    session.display()
    return session


if HEADLESS:
    _missing_main = find_missing_main_figure_patents(df)
    print(f"HEADLESS: {len(_missing_main)} patent(s) would need a main-figure "
          f"review: {_missing_main} (UI skipped)")
else:
    main_figure_session = build_main_figure_review(df)


1 patent(s) need a main-figure review: ['US2018354616A1']


## Section 6 — Export (approved images only)

Writes `Review_postprocess_<batch>_<timestamp>.xlsx` **next to the raw
export** in `data/03_HUMAN_wizard_exports/` (config `html_review_exports`; folder renamed from `reviewed xlsxs/` 2026-08-05), via
`format_review_workbook` (flat `Review` sheet for 02b + merged `Compact`
sheet for humans). The raw `reviewed_patents_<batch>.xlsx` is never touched.

Two scope rules are enforced before the T2-approval filter runs (belt-and-
suspenders with what the wizard itself already omits on export, and with
Rule D's `_inherit_from_duplicate_root` no longer back-filling these):

- **Disapproved patents** (`isApproved == False`) keep only their `T1` rows
  — no G1/M1–M3/T2/META noise from an unreviewed patent.
- **Exact duplicates** (`duplicateType == "2"`, "Images AND aircraft the
  same") keep no `T2`/`G1`/`M1`/`M2`/`M3` rows of their own — the pipeline
  resolves their full labels via `duplicateId` lookup on the original patent.
  "Same Aircraft" duplicates (`duplicateType == "1"`) are unaffected: they
  legitimately carry their own `G1`–`M3` (copied from the original, editable).

After that, T2 image blocks are filtered to **status = approved only** —
disapproved figures, `(fig N)` placeholders and abandoned clipboard pastes
are dropped.


In [19]:
# ── Enforce per-patent row scope ─────────────────────────────────────────────
# Disapproved patents: T1 only. Exact duplicates (duplicateType "2"): no
# T2/G1/M1-M3 of their own (labels come from a duplicateId lookup on the
# original patent — see the Section 6 markdown above and Rule D's
# exact_dup_ids exclusion). "Same Aircraft" duplicates (duplicateType "1")
# keep their own G1-M3 untouched.
appr_rows = df.loc[df["Field"] == "isApproved"]
disapproved_ids = set(
    appr_rows.loc[appr_rows["Value"].astype(str).str.strip().str.lower() == "false", "Patent_ID"].astype(str)
)

dup_type_rows = df.loc[df["Field"] == "duplicateType"]
exact_dup_ids = set(
    dup_type_rows.loc[dup_type_rows["Value"].map(_strip_label) == "2", "Patent_ID"].astype(str)
)

pid_str = df["Patent_ID"].astype(str)
drop_non_t1 = pid_str.isin(disapproved_ids) & (df["Section"] != "T1")
drop_exact_dup_labels = pid_str.isin(exact_dup_ids) & df["Section"].isin(["T2", "G1", "M1", "M2", "M3"])

df = df[~(drop_non_t1 | drop_exact_dup_labels)].reset_index(drop=True)
print(f"scope enforcement: dropped {int(drop_non_t1.sum())} non-T1 row(s) for "
      f"{len(disapproved_ids)} disapproved patent(s) | dropped "
      f"{int(drop_exact_dup_labels.sum())} T2/G1/M1-M3 row(s) for "
      f"{len(exact_dup_ids)} exact-duplicate patent(s)")

# ── Keep only APPROVED image blocks ─────────────────────────────────────────
# A T2 block = all rows sharing (Patent_ID, "Image: <name>" Sub_Dimension).
# Keep a block only if its status row says "approved"; every remaining
# non-T2 row (T1/G1/M1–M3/META labels) is kept unchanged.
t2_status = df[(df["Section"] == "T2") & (df["Field"] == "status")]
approved_blocks = {
    (str(r["Patent_ID"]), str(r["Sub_Dimension"]))
    for _, r in t2_status.iterrows()
    if str(r["Value"]).strip().lower() == "approved"
}
is_t2 = df["Section"] == "T2"
block_key = list(zip(df["Patent_ID"].astype(str), df["Sub_Dimension"].astype(str)))
keep = ~is_t2 | pd.Series([k in approved_blocks for k in block_key], index=df.index)

export_df = df[keep].reset_index(drop=True)
n_dropped_blocks = t2_status.shape[0] - len(approved_blocks)
print(f"approved-only filter: kept {len(export_df)}/{len(df)} rows "
      f"({len(approved_blocks)} approved image blocks, {n_dropped_blocks} blocks dropped)")

# ── Write the new batch file (never overwrites anything) ────────────────────
# 02a's output no longer lands in the frozen human-export folder (2026-08-18):
# that directory holds wizard exports and nothing else. 02b reads from here.
OUTPUT_DIR = Path(cfg["paths"]["cleaned_label_tables"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_XLSX = OUTPUT_DIR / f"Review_postprocess_{sheet_name}_{timestamp}.xlsx"

assert OUTPUT_XLSX != REVIEWED_XLSX, "refusing to overwrite the raw wizard export"
assert not OUTPUT_XLSX.exists(), f"{OUTPUT_XLSX} already exists — refusing to overwrite"

# ── Sanity assertions (fail loudly BEFORE handing the file to 02b) ─────────
_pid = export_df["Patent_ID"].astype(str)
assert len(export_df) > 0, "export is empty — nothing survived the filters; check isApproved/status values"
assert not (_pid.isin(disapproved_ids) & (export_df["Section"] != "T1")).any(), \
    "disapproved patent leaked non-T1 rows into the export"
assert not (_pid.isin(exact_dup_ids)
            & export_df["Section"].isin(["T2", "G1", "M1", "M2", "M3"])).any(), \
    "exact duplicate (type 2) leaked its own label/T2 rows into the export"
_t2_blocks = set(zip(_pid[export_df["Section"] == "T2"],
                     export_df.loc[export_df["Section"] == "T2", "Sub_Dimension"].astype(str)))
assert _t2_blocks <= approved_blocks, "a non-approved T2 block leaked into the export"

format_review_workbook(export_df, OUTPUT_XLSX, truncate_long_text=False)

# ── Round-trip check: what 02b will read back must match what we exported ──
_rt = pd.read_excel(OUTPUT_XLSX, sheet_name="Review")
assert len(_rt) == len(export_df), \
    f"round-trip row-count mismatch: wrote {len(export_df)}, read back {len(_rt)}"
assert _rt["Patent_ID"].notna().all(), "round-trip lost Patent_ID values (merged cells?)"
assert list(_rt.columns) == list(export_df.columns), "round-trip changed the column set"

# ── Final labeller summary ──────────────────────────────────────────────────
print("\n" + "=" * 62)
print(f"  02a DONE — {sheet_name}")
print(f"  input : {REVIEWED_XLSX.name}  ({len(df_raw)} raw rows)")
print(f"  output: {OUTPUT_XLSX.name}")
print(f"  patents in export        : {export_df['Patent_ID'].nunique()}")
print(f"  approved image blocks    : {len(_t2_blocks)}")
print(f"  rows written             : {len(_rt)}  (round-trip verified)")
print(f"  disapproved patents (T1-only): {len(disapproved_ids)}")
print(f"  exact duplicates (label-less): {len(exact_dup_ids)}")
print("  → next: run 02b_postprocessing (it auto-picks this newest file).")
print("  --- addendum (A5/S3/D5) ---")
print(f"  empTilts on rigid arch (TB/TW, eyeball)   : {len(emptilts_rigid_arch_worklist)}")
print(f"  empTilts missing justification (data issue): {len(emptilts_missing_justification)}")
print(f"  confidence field(s) found                 : {confidence_constancy_report['wide_columns'] or ('Field==confidence' if confidence_constancy_report['field_rows'] else 'none')}")
print("=" * 62)



scope enforcement: dropped 120 non-T1 row(s) for 60 disapproved patent(s) | dropped 0 T2/G1/M1-M3 row(s) for 167 exact-duplicate patent(s)
approved-only filter: kept 22684/25364 rows (301 approved image blocks, 1443 blocks dropped)
Successfully generated formatted workbook (Review + Compact): /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/reviewed xlsxs/Review_postprocess_Batch_01_20260706_141550.xlsx

  02a DONE — Batch_01
  input : reviewed_patents_Batch_01.xlsx  (25584 raw rows)
  output: Review_postprocess_Batch_01_20260706_141550.xlsx
  patents in export        : 393
  approved image blocks    : 301
  rows written             : 22684  (round-trip verified)
  disapproved patents (T1-only): 60
  exact duplicates (label-less): 167
  → next: run 02b_postprocessing (it auto-picks this newest file).
